In [ ]:
import sys
sys.path.insert(0, '../../stock_factor_lab_2025/')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# 台股實驗彙總

## 回測期間設定

In [ ]:
START_DATE = '2003-3-31'
END_DATE = '2024-12-31'

## 前置作業

### import

In [ ]:
from get_data import Data
import backtest
from combinations import sim_conditions
import random
import talib
import pandas as pd
import numpy as np
from datetime import datetime
from dateutil.relativedelta import relativedelta
import itertools

import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from itertools import cycle
from plotly.subplots import make_subplots
import matplotlib as mpl
from matplotlib.ticker import FuncFormatter
import matplotlib.dates as mdates
import seaborn as sns
import re

from matplotlib import rcParams
rcParams['font.sans-serif'] = ['Microsoft JhengHei']
mpl.rcParams['axes.unicode_minus'] = False

from dataframe import CustomDataFrame

### get data

In [ ]:
data=Data()

## 資料下載

In [ ]:
profit = data.get('annual_report_fundamentals:常續性稅後淨利')
income_bf_tax = data.get('annual_report_fundamentals:稅前淨利')

# 本益比
pe = data.get('quarter_report:PE')
daily_pe = data.get('price:daily_pe')

roe = data.get('annual_report:ROE')

payout_ratio = data.get('annual_report_fundamentals:股利支付率')

stock_hold = data.get('annual_report_fundamentals:董監持股%')

In [ ]:
long_term_items = [
    '透過損益按公允價值衡量之金融資產－非流動', #2005
    '透過其他綜合損益按公允價值衡量之金融資產－非流動', #2005
    '按攤銷後成本衡量之金融資產－非流動', #2005
    '避險之金融資產－非流動', #2005
    '合約資產－非流動', #2017
    '採權益法之長期股權投資',
    '預付投資款', #2005
    '投資性不動產淨額'
]
# 長期投資項目 (8個)
long_term_data = [data.get(f'annual_report_fundamentals:{item}').fillna(0) for item in long_term_items]
long_term_investment = sum(long_term_data)
# 固定資產
fixed_assets_year = data.get('annual_report_fundamentals:不動產廠房及設備').fillna(0)
# 計算盈再率分子
long_term_investment_df = (long_term_investment - long_term_investment.shift(4))
fixed_assets_df = (fixed_assets_year - fixed_assets_year.shift(4))
# 分母
profit_rol_df = profit.rolling(4).sum()

In [ ]:
# 計算出盈再率
# 去掉淨利為負的公司 (當年度淨利為負、四年加總淨利為負都去掉)
# 因為2005年之前財報有改制，所以在這之前用非流動資產來算分子，但其實好像跟全部用改制後的方法來算差不多（洪瑞泰盈再表的分子項目沒有包含所有長期投資項）
orig_rr = (long_term_investment_df + fixed_assets_df) / profit_rol_df
orig_rr = orig_rr[(profit > 0) & (profit_rol_df > 0)]['2006':]

rr_sum = data.get('annual_report_fundamentals:非流動資產')
rr_sum_df = rr_sum - rr_sum.shift(4)

old_rr = rr_sum_df / profit_rol_df
old_rr = old_rr[(profit > 0) & (profit_rol_df > 0)][START_DATE:'2005']

In [ ]:
common_columns = sorted(list(set(orig_rr.columns) & set(old_rr.columns)))
# 合併資料
rr = pd.concat([old_rr[common_columns], orig_rr[common_columns]])

In [ ]:
close = data.get('price:close')

comp_profile = data.get('company_profile')
list_stock_data = {}
for index, row in comp_profile.iterrows():
    stock_code = row['company_symbol']
    listed_date = row['ipo_date']
    end_date = listed_date + pd.DateOffset(years=2)
    
    # 創建一個全為 True 的 series
    series = pd.Series(True, index=close.index)
    # 在上市日之前和之後兩年內設置為 False
    series.loc[:end_date] = False
    list_stock_data[stock_code] = series

listed = pd.concat(list_stock_data, axis=1)
listed = CustomDataFrame(listed)

## 原始條件

In [ ]:
roe_rol = roe.rolling(5).mean()
roe_15 = roe_rol > 15

rr_cond = rr < 0.4

payout_ratio_cond = (payout_ratio.rolling(3).min() >= 40)

profit_cond = profit > 500000 # TEJ 的淨利單位是千元

hold_cond = stock_hold > 10

listed = listed.resample('M').last()

In [ ]:
# 每季本益比
pe_entry = pe < 12
pe_exit = pe > 30

# 每月本益比
# 每日本益比resample成每月
daily_pe_resample = daily_pe.resample('M').last()

pe_cond_entry_daily = (daily_pe < 12).resample('M').last()
pe_cond_exit_daily = (daily_pe > 30).resample('M').last()

### 台股各選股原則滿足條件的公司比例

In [ ]:
stock_data = {}
for index, row in comp_profile.iterrows():
    stock_code = row['company_symbol']
    listed_date = row['ipo_date']
    
    # 創建一個全為 True 的 series
    series = pd.Series(True, index=close.index)

    series.loc[:listed_date] = False
    stock_data[stock_code] = series

listed_comp_df = pd.concat(stock_data, axis=1)

In [ ]:
fig = plt.figure(figsize=(14, 6))

# 準備資料
roe_data = (roe_15[START_DATE:END_DATE].sum(axis=1) / listed_comp_df[START_DATE:END_DATE].sum(axis=1).resample('A-MAR').last()) * 100
rr_data = (rr_cond[START_DATE:END_DATE].sum(axis=1) / listed_comp_df[START_DATE:END_DATE].sum(axis=1).resample('A-MAR').last()) * 100
payout_data = (payout_ratio_cond[START_DATE:END_DATE].sum(axis=1) / listed_comp_df[START_DATE:END_DATE].sum(axis=1).resample('A-MAR').last()) * 100
profit_data = (profit_cond[START_DATE:END_DATE].sum(axis=1) / listed_comp_df[START_DATE:END_DATE].sum(axis=1).resample('A-MAR').last()) * 100
# common_columns = hold_cond.columns.intersection(payout_ratio_cond.columns)
hold_cond_data = ((hold_cond & listed)[START_DATE:END_DATE].sum(axis=1) / listed_comp_df[START_DATE:END_DATE].sum(axis=1).resample('M').last()) * 100
listed_data = (listed[START_DATE:END_DATE].sum(axis=1) / listed_comp_df[START_DATE:END_DATE].sum(axis=1).resample('M').last()) * 100
pe_cond_entry_data = (pe_cond_entry_daily[START_DATE:END_DATE].sum(axis=1) / listed_comp_df[START_DATE:END_DATE].sum(axis=1).resample('M').last()) * 100
pe_cond_exit_data = (pe_cond_exit_daily[START_DATE:END_DATE].sum(axis=1) / listed_comp_df[START_DATE:END_DATE].sum(axis=1).resample('M').last()) * 100

# 繪製折線圖
plt.plot(roe_data.index, roe_data, marker='o', linestyle='-', markersize=3, label='ROE五年平均 > 15%')
plt.plot(rr_data.index, rr_data, marker='s', linestyle='--', markersize=3, label='盈餘再投資率 < 40%')
plt.plot(payout_data.index, payout_data, marker='^', linestyle='-.', markersize=3, label='股利支付率三年至少 40%')
plt.plot(profit_data.index, profit_data, marker='x', linestyle=':', markersize=3, label='稅後淨利 > 五億台幣')
plt.plot(hold_cond_data.index, hold_cond_data, marker='D', linestyle='--', markersize=1, label='董監持股% > 10%')
plt.plot(listed_data.index, listed_data, marker='|', linestyle='-', markersize=1, label='上市櫃滿兩年')
plt.plot(pe_cond_entry_data.index, pe_cond_entry_data, marker='*', linestyle='-', markersize=3, label='本益比 < 12 (進場條件)')
plt.plot(pe_cond_exit_data.index, pe_cond_exit_data, marker='>', linestyle='--', markersize=3, label='本益比 > 30 (出場條件)')

# 設置 X 軸為每年年份
plt.xlabel("Year", fontsize=16)
plt.ylabel("占比 (%)", fontsize=16)
plt.title("2003~2024 台股各選股原則滿足條件占所有上市櫃公司的比例", fontsize=16)
plt.grid(True)
plt.xticks(
    ticks=pd.date_range(start="2003-01-01", end="2024-12-31", freq="YS"),
    labels=pd.date_range(start="2003-01-01", end="2024-12-31", freq="YS").strftime("%Y"),
    rotation=45,
    fontsize=14
)

plt.grid(linestyle='--', linewidth=0.5, alpha=0.6)
plt.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=14)  # 將圖例移動到圖表右上角外
plt.tight_layout()
plt.show()

fig.savefig('./img/圖 35 台股市場中符合各項選股條件的公司數占所有上市櫃公司數的比例.svg', format='svg', dpi=1200)

---

## 回測

In [ ]:
# 原始條件
orig_all_cond = (roe_15 & rr_cond & payout_ratio_cond & profit_cond & hold_cond & listed)[START_DATE:END_DATE]
# 原始條件 + 每季本益比
orig_all_cond_and_pe = ((orig_all_cond & pe_entry[START_DATE:END_DATE]).hold_until((~orig_all_cond) | pe_exit[START_DATE:END_DATE]))
# 原始條件 + 每月本益比
orig_all_cond_and_pe_daily = ((orig_all_cond & pe_cond_entry_daily[START_DATE:END_DATE]).hold_until((~orig_all_cond) | pe_cond_exit_daily[START_DATE:END_DATE]))

In [ ]:
rep_all_cond_dic = {}

rep_all_cond_dic['台股_原始條件_不含本益比進出場'] = orig_all_cond
rep_all_cond_dic['台股_原始條件_每季本益比進出場'] = orig_all_cond_and_pe
rep_all_cond_dic['台股_原始條件_每月月底本益比進出場'] = orig_all_cond_and_pe_daily

In [ ]:
rep_all_cond = sim_conditions(rep_all_cond_dic, resample='M', data=data)

In [ ]:
# rep_all_cond.plot_creturns()

In [ ]:
# rep_all_cond.plot_stats()

In [ ]:
rep_all_cond.selected_stock_count_analysis()

In [ ]:
rep_all_cond_df = rep_all_cond.selected_stock_count_analysis()
rep_all_cond_df = rep_all_cond_df.reset_index()

# 分組資料
no_pe = rep_all_cond_df[rep_all_cond_df['Strategy'].str.contains('不含本益比進出場')]
monthly_pe = rep_all_cond_df[rep_all_cond_df['Strategy'].str.contains('月底本益比進出場')]

# 提取x軸標籤
no_pe_labels = [s.split('_')[1:2] for s in no_pe['Strategy']]
no_pe_labels = ['_'.join(label) for label in no_pe_labels]

In [ ]:
# 設置圖表
fig = plt.figure(figsize=(9, 8))
width = 0.04  # bar的寬度
group_padding = 0.1  # 不同組之間的間距

# 設置x軸位置
x = range(len(no_pe_labels))

# 繪製bars
bars1 = plt.bar([i - group_padding for i in x], no_pe['CAGR (%)'], width, label='CAGR', color='tab:blue', alpha=0.7)
bars2 = plt.bar([i - group_padding + width for i in x], no_pe['MDD (%)'], width, label='MDD', color='orange', alpha=0.7)
bars3 = plt.bar([i + group_padding - width for i in x], monthly_pe['CAGR (%)'], width, color='tab:blue', alpha=0.7)
bars4 = plt.bar([i + group_padding for i in x], monthly_pe['MDD (%)'], width, color='orange', alpha=0.7)

# 添加數值標籤
for bar in bars1 + bars2 + bars3 + bars4:
    height = bar.get_height()
    if height > 0:
        plt.text(bar.get_x() + bar.get_width() / 2, height, f'{height:.2f}', ha='center', va='bottom', fontsize=10)
    else:
        plt.text(bar.get_x() + bar.get_width() / 2, height, f'{height:.2f}', ha='center', va='top', fontsize=10)

# 獲取軸對象
ax = plt.gca()

# 隱藏主要x軸標籤
ax.set_xticklabels([])

# 添加次要x軸標籤（本益比條件）
ax2 = ax.secondary_xaxis('bottom') 
# 調整標籤位置，使其對齊對應的柱狀圖組
ax2.set_xticks([i - group_padding + width/2 for i in x] + [i + group_padding - width/2 for i in x])
ax2.set_xticklabels(['不含本益比進出場']*len(no_pe_labels) + ['每月本益比進出場']*len(no_pe_labels), fontsize=14)

# 設置Y軸刻度，以10為間隔
ax.yaxis.set_major_locator(plt.MultipleLocator(10))

# 設置網格線
plt.grid(True, alpha=0.4, which='both', axis='y')

plt.ylabel('百分比', fontsize=12)
plt.title('台股 2003-2024 符合所有條件_有無本益比進出場_每月換股', fontsize=14)
plt.legend(['CAGR', 'MDD'], loc='lower center', ncol=2, fontsize=12)
plt.axhline(0, color='red', linewidth=0.5)

# 顯示圖表
plt.show()

fig.savefig('./img/圖 4台股原始策略CAGR與MDD比較圖.svg', dpi=300, format='svg', bbox_inches='tight')

In [ ]:
# # 入選股數占比平均%
# rep_all_cond.selected_stock_count_analysis(ratio=True)

In [ ]:
fig = rep_all_cond.plot_reps_stock_counts(['台股_原始條件_不含本益比進出場', '台股_原始條件_每月月底本益比進出場'])
fig.savefig('./img/圖 5台股原始策略入選股數變化圖.svg', dpi=300, format='svg', bbox_inches='tight')

In [ ]:
# orig_pe_rep_period_df = rep_all_cond.reports['台股_原始條件_每月月底本益比進出場'].trades
# orig_pe_rep_period_df = orig_pe_rep_period_df.reset_index(inplace=False).sort_values(by='period', ascending=False, inplace=False)
# orig_pe_rep_period_df.head(20)

In [ ]:
# orig_rep_period_df = rep_all_cond.reports['台股_原始條件_不含本益比進出場'].trades
# orig_rep_period_df = orig_rep_period_df.reset_index(inplace=False).sort_values(by='period', ascending=False, inplace=False)
# orig_rep_period_df.head(20)

---

In [ ]:
rep_all_cond.reports['台股_原始條件_每月月底本益比進出場'].display()

### 切分時間段2003~2009、2009~2024

In [ ]:
orig_all_cond_2003_2009 = (roe_15 & rr_cond & payout_ratio_cond & profit_cond & hold_cond & listed)[START_DATE:'2009-3-31']
orig_all_cond_2009_2024 = (roe_15 & rr_cond & payout_ratio_cond & profit_cond & hold_cond & listed)['2009-3-31':END_DATE]

# orig_all_cond_and_pe_2003_2009 = ((orig_all_cond_2003_2009 & pe_entry[START_DATE:'2009-3-31']).hold_until((~orig_all_cond_2003_2009) | pe_exit[START_DATE:'2009-3-31']))
# orig_all_cond_and_pe_2009_2024 = ((orig_all_cond_2009_2024 & pe_entry['2009-3-31':END_DATE]).hold_until((~orig_all_cond_2009_2024) | pe_exit['2009-3-31':END_DATE]))

orig_all_cond_and_pe_daily_2003_2009 = ((orig_all_cond_2003_2009 & pe_cond_entry_daily[START_DATE:'2009-3-31']).hold_until((~orig_all_cond_2003_2009) | pe_cond_exit_daily[START_DATE:'2009-3-31']))
orig_all_cond_and_pe_daily_2009_2024 = ((orig_all_cond_2009_2024 & pe_cond_entry_daily['2009-3-31':END_DATE]).hold_until((~orig_all_cond_2009_2024) | pe_cond_exit_daily['2009-3-31':END_DATE]))

In [ ]:
time_period_dic = {}

time_period_dic['台股_原始條件_不含本益比進出場_2003-2009'] = orig_all_cond_2003_2009
time_period_dic['台股_原始條件_不含本益比進出場_2009-2024'] = orig_all_cond_2009_2024


# time_period_dic['台股_原始條件_每季本益比進出場_2003-2009'] = orig_all_cond_and_pe_2003_2009
# time_period_dic['台股_原始條件_每季本益比進出場_2009-2024'] = orig_all_cond_and_pe_2009_2024

time_period_dic['台股_原始條件_每月月底本益比進出場_2003-2009'] = orig_all_cond_and_pe_daily_2003_2009
time_period_dic['台股_原始條件_每月月底本益比進出場_2009-2024'] = orig_all_cond_and_pe_daily_2009_2024

time_period_rep_collec = sim_conditions(time_period_dic, resample="M", data=data)

In [ ]:
# time_period_rep_collec.plot_creturns()

In [ ]:
time_period_rep_collec.selected_stock_count_analysis()

In [ ]:
def perform_compare_barchart(df, colors=None, hatches=None):
    df.reset_index(inplace=True)
    # 提取策略名稱
    df['base_strategy'] = df['Strategy'].apply(lambda x: x.rsplit('_', 1)[0])
    # 提取時間段
    df['time_period'] = df['Strategy'].apply(lambda x: x.rsplit('_', 1)[1])
    
    # 定義策略順序
    strategy_order = {
        '台股_原始條件_不含本益比進出場': 0,
        '台股_原始條件_每季本益比進出場': 1,
        '台股_原始條件_每月月底本益比進出場': 2
    }
    
    # X軸策略按照指定順序排列呈現
    unique_strategies = sorted(df['base_strategy'].unique(), 
                             key=lambda x: strategy_order[x])
    time_periods = sorted(df['time_period'].unique())
    
    fig = plt.figure(figsize=(15, 6))

    width = 0.2  # 增加 bar 的寬度
    spacing = 0.05  # 增加 bar group 之間的間距
    x = np.arange(len(unique_strategies)) * (1 + spacing)  # 調整 x 軸位置
    
    # 如果沒有提供顏色，使用默認顏色
    if colors is None:
        colors = ['tab:blue', 'tab:orange', 'tab:green']
    
    # 如果沒有提供hatch patterns，使用默認hatch patterns
    if hatches is None:
        hatches = ['', '', ''] # ['', '//', 'xx']
    
    # CAGR比較圖
    ax1 = plt.subplot(1, 2, 1)
    bars1 = []  # 存儲條形對象用於後面的legend
    for i, period in enumerate(time_periods):
        period_data = df[df['time_period'] == period]
        # 根據策略順序重新排序數據
        period_data = period_data.sort_values(by='base_strategy', 
                                            key=lambda x: x.map(strategy_order))
        bars1.append(ax1.bar(x + i*width, period_data['CAGR (%)'], width, alpha=0.8, color=colors[i % len(colors)], hatch=hatches[i % len(hatches)]))
    
    ax1.set_xlabel('Strategy')
    ax1.set_ylabel('CAGR (%)', fontsize=12)
    ax1.set_title('台股 2009-2009、2009-2024 不同時間段 CAGR 比較', fontsize=15)
    ax1.set_xticks(x + width/2)
    ax1.set_xticklabels([s.split('_')[2] for s in unique_strategies], rotation=0, fontsize=12)
    ax1.grid(True, alpha=0.3)
    ax1.tick_params(axis='y', labelsize=13)
    
    # MDD比較圖
    ax2 = plt.subplot(1, 2, 2)
    bars2 = []  # 存儲條形對象用於後面的legend
    for i, period in enumerate(time_periods):
        period_data = df[df['time_period'] == period]
        period_data = period_data.sort_values(by='base_strategy', 
                                            key=lambda x: x.map(strategy_order))
        bars2.append(ax2.bar(x + i*width, period_data['MDD (%)'], width, alpha=0.8, color=colors[i % len(colors)], hatch=hatches[i % len(hatches)]))
    
    ax2.set_xlabel('Strategy')
    ax2.set_ylabel('MDD (%)', fontsize=12)
    ax2.set_title('台股 2009-2009、2009-2024 不同時間段 MDD 比較', fontsize=15)
    ax2.set_xticks(x + width/2)
    ax2.set_xticklabels([s.split('_')[2] for s in unique_strategies], rotation=0, fontsize=12)
    ax2.grid(True, alpha=0.3)
    ax2.tick_params(axis='y', labelsize=13)
    
    # 調整布局
    plt.tight_layout()

    # 在底部中間添加legend
    fig.legend(bars1, time_periods, 
              loc='center', 
              bbox_to_anchor=(0.5, 0.02),
              ncol=len(time_periods),
              fontsize=13)
    
    plt.subplots_adjust(bottom=0.2)  # 為底部的legend留出空間

    plt.show()

    fig.savefig('./img/圖 11台股原始策略CAGR與MDD在兩段時期的差異比較圖.svg', dpi=300, format='svg', bbox_inches='tight')


In [ ]:
time_period_tests_df = time_period_rep_collec.selected_stock_count_analysis()
perform_compare_barchart(time_period_tests_df)

In [ ]:
# time_period_rep_collec.reports['台股_原始條件_每月月底本益比進出場_2003-2009'].display()

In [ ]:
# def plot_strategy_cumm_return(rep, title='累積報酬'):
#         """繪製單一策略累積報酬率走勢圖

#             Args:
#                 rep (report): 回測report物件
#                 title (str): 圖表標題

#             Returns:
#                 None: 顯示策略累積報酬率走勢圖
#         """
        
#         cum_returns = rep.stock_data['cum_returns']
#         tw_bench = rep.data.get('taiex:close')
#         tw_close = tw_bench['close'].reindex(cum_returns.index)
#         tw_close = tw_close / tw_close.iloc[0]

#         plt.figure(figsize=(18, 6))
#         plt.plot(cum_returns.index, cum_returns, label='策略累積報酬')
#         plt.plot(tw_close.index, tw_close, label='台股大盤', color='gray', alpha=0.4)

#         plt.xlabel('年份')
#         plt.ylabel('百分比')
#         plt.axhline(1, color='red', linewidth=0.5)
#         plt.title(f'{title}', fontsize=16)
#         plt.legend(fontsize=14)
#         plt.grid(True, alpha=0.3, linestyle='--')

#         # Custom y-ticks formatter
#         plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, pos: f'{(x - 1) * 100:.0f}%'))

#         plt.show()

In [ ]:
time_period_rep_collec.reports['台股_原始條件_每月月底本益比進出場_2003-2009'].plot_strategy_cumm_return(title='台股_原始條件_每月月底本益比進出場_2003-2009_累積報酬')

In [ ]:
rep_all_cond.reports['台股_原始條件_每月月底本益比進出場'].plot_strategy_cumm_return(title='台股_原始條件_每月月底本益比進出場_2003-2024_累積報酬')

In [ ]:
# tw_bench = data.get('taiex:close')[START_DATE:END_DATE]
# tw_bench['close'] = tw_bench['close'] / tw_bench['close'].iloc[0]

In [ ]:
# cum_returns = time_period_rep_collec.reports['台股_原始條件_每月月底本益比進出場_2003-2009'].stock_data['cum_returns']
# tw_close = tw_bench['close'].reindex(cum_returns.index)

# plt.figure(figsize=(18, 6))

# # Plot cumulative returns
# plt.plot(cum_returns.index, cum_returns, label='台股_原始條件_每月月底本益比進出場_2003-2009_累積報酬')

# # Plot normalized close prices
# plt.plot(tw_close.index, tw_close, label='台股大盤', color='gray', alpha=0.4)

# plt.xlabel('年份')
# plt.ylabel('百分比')
# plt.axhline(1, color='red', linewidth=0.5)
# plt.title('累積報酬', fontsize=16)
# plt.legend(fontsize=14)
# plt.grid(True, alpha=0.3, linestyle='--')

# # Custom y-ticks formatter
# plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, pos: f'{(x - 1) * 100:.0f}%'))

# plt.show()

In [ ]:
# cum_returns = rep_all_cond.reports['台股_原始條件_每月月底本益比進出場'].stock_data['cum_returns']
# tw_close = tw_bench['close'].reindex(cum_returns.index)

# plt.figure(figsize=(18, 6))

# # Plot cumulative returns
# plt.plot(cum_returns.index, cum_returns, label='台股_原始條件_每月月底本益比進出場_2003-2024_累積報酬')

# # Plot normalized close prices
# plt.plot(tw_close.index, tw_close, label='台股大盤', color='gray', alpha=0.4)

# plt.xlabel('年份')
# plt.ylabel('百分比')
# plt.axhline(1, color='red', linewidth=0.5)
# plt.title('累積報酬', fontsize=16)
# plt.legend(fontsize=14)
# plt.grid(True, alpha=0.3, linestyle='--')

# # Custom y-ticks formatter
# plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, pos: f'{(x - 1) * 100:.0f}%'))

# plt.show()

## 獲利分佈

In [ ]:
# orig_strat_rep_test = rep_all_cond.reports['台股_原始條件_每月月底本益比進出場']

# pos_df = orig_strat_rep_test.position
# close_price_df = close[START_DATE:END_DATE]

In [ ]:
# monthly_first = close_price_df.resample('M').first()
# monthly_last = close_price_df.resample('M').last()

# # 計算股價每月的增長率：(月底股價/月初股價) - 1
# monthly_returns_df = (monthly_last / monthly_first) - 1

In [ ]:
# # 每月資金分配比例 * 每月股價增長率
# returns_pos_df = monthly_returns_df * pos_df.shift() # 向下移動一個row

In [ ]:
# # 轉換成年度數據，將各股在同一年的每月報酬率相加
# returns_pos_df_annual = returns_pos_df.resample('A').sum()
# # returns_pos_df_annual.sum(axis=1)

In [ ]:
# def create_stacked_returns_plot(returns_pos_df_annual, top_N=None):
#     from itertools import cycle

#     # 年份列表
#     years = returns_pos_df_annual.index.year.tolist()

#     # 為每支股票生成唯一的顏色
#     stocks = returns_pos_df_annual.columns.tolist()
#     n_stocks = len(stocks)

#     # 使用 plotly 的顏色配色方案
#     all_colors = px.colors.qualitative.Plotly + px.colors.qualitative.Set1 + px.colors.qualitative.Pastel + px.colors.qualitative.Dark2
#     color_cycle = cycle(all_colors)
#     color_dict = {stock: next(color_cycle) for stock in stocks}

#     # 創建用於正收益和負收益的空列表
#     positive_traces = []
#     negative_traces = []

#     # 針對df中每支個股進行處理
#     for stock in returns_pos_df_annual.columns:
#         stock_returns = returns_pos_df_annual[stock]

#         # 分離正收益和負收益
#         positive_returns = stock_returns.copy()
#         positive_returns[positive_returns <= 0] = 0

#         negative_returns = stock_returns.copy()
#         negative_returns[negative_returns >= 0] = 0

#         # 如果有正收益，創建正收益trace
#         if positive_returns.max() > 0:
#             for year_idx, value in enumerate(positive_returns):
#                 if value > 0:  # 只添加正值
#                     positive_traces.append({
#                         'stock': stock,
#                         'year_idx': year_idx,
#                         'year': years[year_idx],
#                         'return': value,
#                         'color': color_dict[stock]
#                     })

#         # 如果有負收益，創建負收益trace
#         if negative_returns.min() < 0:
#             for year_idx, value in enumerate(negative_returns):
#                 if value < 0:  # 只添加負值
#                     negative_traces.append({
#                         'stock': stock,
#                         'year_idx': year_idx,
#                         'year': years[year_idx],
#                         'return': value,
#                         'color': color_dict[stock]
#                     })

#     # 創建圖表
#     fig = go.Figure()

#     # 按年份分組並排序正收益
#     for year in years:
#         year_traces = [t for t in positive_traces if t['year'] == year]
#         # 按報酬值從大到小排序
#         year_traces.sort(key=lambda x: x['return'], reverse=True)

#         if top_N is not None and len(year_traces) > top_N:
#             # 分離前N名和其他
#             top_traces = year_traces[:top_N]
#             other_traces = year_traces[top_N:]

#             # 計算"其他"區塊的正報酬的總和
#             other_sum = sum(trace['return'] for trace in other_traces)

#             # 繪製"其他"區塊在正報酬bar stack的最底下
#             if other_sum > 0:
#                 fig.add_trace(go.Bar(
#                     name=f"其他 ({year})",
#                     x=[year],
#                     y=[other_sum],
#                     base=[0],
#                     marker=dict(
#                         color='white',
#                         pattern_shape="/",
#                     ),
#                     showlegend=False,
#                     hovertemplate=f'%{{x}}<br>其他: %{{y:.2%}}<extra></extra>'
#                 ))

#             # 再繪製前N名，堆疊在`其他`區塊的上方
#             cumulative = other_sum
#             for trace in reversed(top_traces):
#                 fig.add_trace(go.Bar(
#                     name=trace['stock'],
#                     x=[year],
#                     y=[trace['return']],
#                     base=[cumulative],
#                     marker_color=trace['color'],
#                     showlegend=False,
#                     hovertemplate=f'{trace["stock"]}: {trace["return"]:.2%}<extra></extra>'
#                 ))
#                 cumulative += trace['return']
#         else:
#             # 全部顯示（從底部開始堆疊）
#             cumulative = 0
#             for trace in reversed(year_traces):
#                 fig.add_trace(go.Bar(
#                     name=trace['stock'],
#                     x=[year],
#                     y=[trace['return']],
#                     base=[cumulative],
#                     marker_color=trace['color'],
#                     showlegend=False,
#                     hovertemplate=f'{trace["stock"]}: {trace["return"]:.2%}<extra></extra>'
#                 ))
#                 cumulative += trace['return']

#     # 按年份分組並排序負收益
#     for year in years:
#         year_traces = [t for t in negative_traces if t['year'] == year]
#         # 按報酬值從大到小排序（負值從小到大）
#         year_traces.sort(key=lambda x: x['return'], reverse=True)

#         cumulative = 0
#         for trace in year_traces:
#             fig.add_trace(go.Bar(
#                 name=trace['stock'],
#                 x=[year],
#                 y=[trace['return']],
#                 base=[cumulative],
#                 marker_color=trace['color'],
#                 showlegend=False,
#                 hovertemplate=f'{trace["stock"]}: {trace["return"]:.2%}<extra></extra>'
#             ))
#             cumulative += trace['return']

#     # 添加唯一的圖例條目
#     unique_stocks = sorted(set(trace['stock'] for trace in positive_traces + negative_traces))
#     for stock in unique_stocks:
#         fig.add_trace(go.Bar(
#             name=stock,
#             x=[None],
#             y=[None],
#             marker_color=color_dict[stock],
#             showlegend=True,
#             hoverinfo='skip'
#         ))

#     fig.add_trace(go.Bar(
#         name="其他",
#         x=[None],
#         y=[None],
#         marker=dict(
#             color='white',
#             pattern_shape="/",
#         ),
#         showlegend=True,
#         hoverinfo='skip'
#     ))

#     fig.update_layout(
#         barmode='relative',
#         title='年度股票報酬分布',
#         xaxis_title='年份',
#         yaxis_title='報酬',
#         yaxis_tickformat=".0%",
#         xaxis=dict(
#             tickmode='array',
#             tickvals=list(range(min(years), max(years) + 1)),
#             ticktext=list(range(min(years), max(years) + 1))
#         ),
#         showlegend=True,
#         legend_title='股票',
#         hovermode='closest',
#         width=1400,  # 調整寬度以顯示所有年份
#         height=600,
#     )

#     # 添加水平線
#     fig.add_hline(y=0, line_width=1, line_dash="solid", line_color="black")

#     return fig

In [ ]:
# create_stacked_returns_plot(returns_pos_df_annual)

In [ ]:
# create_stacked_returns_plot(returns_pos_df_annual, 5)
# fig.to_image(format="svg", width=1400, height=600, scale=2)

---

## 台股_布林通道濾網

In [ ]:
# 大盤股價
taiex_close = data.get('taiex:close')['2000-3':END_DATE]
# 布林通道上中下通道
upperband, middleband, lowerband = talib.BBANDS(taiex_close.close, timeperiod=300, nbdevup=2.0, nbdevdn=2.0)

### 建立濾網

In [ ]:
# 創建一個買賣訊號的DataFrame，初始值全部為True
tw_bollinger_signal = pd.Series(True, index=taiex_close.index)

# 記錄第一次跌破的價格
first_break_price = None
# 記錄是否已經回到lower band之上
crossed_above_lower = False
# 是否進入持續的 False 狀態直到突破中通道
in_selling_state = False

# 遍歷所有的日期
for date in taiex_close.index:
    price = taiex_close.close[date]
    lower = lowerband[date]
    middle = middleband[date]
    
    # 第一次跌破下通道
    if price < lower and first_break_price is None:
        first_break_price = price
        crossed_above_lower = False
    
    # 價格回到下通道之上
    elif price > lower:
        crossed_above_lower = True
    
    # 再次跌破下通道且符合賣出條件
    if price < lower and crossed_above_lower and price < first_break_price:
        
        tw_bollinger_signal[date] = False
        in_selling_state = True
    
    # 突破中通道則重置狀態
    if price > middle:
        tw_bollinger_signal[date] = True
        first_break_price = None
        crossed_above_lower = False
        in_selling_state = False
    
    # 在突破中通道之前保持賣出狀態
    if in_selling_state and price <= middle:
        tw_bollinger_signal[date] = False

In [ ]:
# tw_bollinger_signal_1.to_csv('./performance_file/TW/taiex_bollinger_trade_signal_all.csv')

### 台股布林通道濾網_視覺化

In [ ]:
# plt.figure(figsize=(20, 6))

# plt.plot(upperband[START_DATE:END_DATE],label="upperband",color='r',linestyle='solid', linewidth=1)
# plt.plot(middleband[START_DATE:END_DATE],label="middleband",color='g',linestyle='--', linewidth=1)
# plt.plot(lowerband[START_DATE:END_DATE],label="lowerband",color='b',linestyle='solid', linewidth=1)
# plt.plot(taiex_close[START_DATE:END_DATE],label="台股大盤",color='black', linewidth=0.8)

# # 出場時間段
# plt.axvspan(pd.Timestamp('2008-7-25'), pd.Timestamp('2009-5-5'), color='gray', alpha=0.25)
# plt.axvspan(pd.Timestamp('2011-9-14'), pd.Timestamp('2012-3-1'), color='gray', alpha=0.25)
# plt.axvspan(pd.Timestamp('2015-9-10'), pd.Timestamp('2016-6-6'), color='gray', alpha=0.25)
# plt.axvspan(pd.Timestamp('2018-12-6'), pd.Timestamp('2019-3-19'), color='gray', alpha=0.25)
# plt.axvspan(pd.Timestamp('2022-6-20'), pd.Timestamp('2023-3-3'), color='gray', alpha=0.25)

# plt.title("台股大盤布林通道（MA300 標準差=2）", fontsize=16) 
# plt.xlabel("年份", fontsize=14) 
# plt.xticks(fontsize=14)
# plt.ylabel("台股大盤")
# plt.yticks(fontsize=14)

# plt.legend(fontsize=14)
# plt.grid(True, linestyle='--', alpha=0.4)
# plt.show()

In [ ]:
fig = plt.figure(figsize=(20, 6))

# 繪製基本圖表
plt.plot(upperband[START_DATE:END_DATE], label="upperband", color='r', linestyle='solid', linewidth=1)
plt.plot(middleband[START_DATE:END_DATE], label="middleband", color='g', linestyle='--', linewidth=1)
plt.plot(lowerband[START_DATE:END_DATE], label="lowerband", color='b', linestyle='solid', linewidth=1)
plt.plot(taiex_close[START_DATE:END_DATE], label="台股大盤", color='black', linewidth=0.8)

# 定義灰底區間及其標籤
spans = [
    ('2008-7-25', '2009-5-5'),
    ('2011-9-14', '2012-3-1'),
    ('2015-9-10', '2016-6-6'),
    ('2018-12-6', '2019-3-19'),
    ('2022-6-20', '2023-3-3')
]

# 繪製灰底和添加標籤
for start, end in spans:
    # 添加灰底
    plt.axvspan(pd.Timestamp(start), pd.Timestamp(end), color='gray', alpha=0.25)
    
    # 計算區間中點位置(用於放置標籤)
    mid_point = pd.Timestamp(start) + (pd.Timestamp(end) - pd.Timestamp(start))/2
    
    # 添加日期區間標籤
    label_text = f'{start}~{end}'
    plt.text(mid_point, plt.ylim()[1]*1.05, label_text,
             horizontalalignment='center',
             fontsize=12)

plt.title("台股大盤布林通道（MA300 標準差=2）", fontsize=16, pad=40)  # 增加 pad 來留出標籤空間
plt.xlabel("年份", fontsize=14)
plt.xticks(fontsize=14)
plt.ylabel("台股大盤")
plt.yticks(fontsize=14)

plt.legend(fontsize=14)
plt.grid(True, linestyle='--', alpha=0.4)

# 調整上方邊界以容納標籤
plt.margins(y=0.1)

plt.show()

# fig.savefig('./img/圖 16台股大盤布林通道與出場時間示意圖.svg', dpi=300, format='svg', bbox_inches='tight')

### 範例圖

In [ ]:
fig = plt.figure(figsize=(20, 6))

# 2008-6-30第一次跌破下通道
# pd.Timestamp('2008-7-25'), pd.Timestamp('2009-5-5')

plt.plot(upperband['2008-6-1':'2009-5-15'], label="upperband", color='r', linestyle='solid', linewidth=1)
plt.plot(middleband['2008-6-1':'2009-5-15'], label="middleband", color='g', linestyle='--', linewidth=1)
plt.plot(lowerband['2008-6-1':'2009-5-15'], label="lowerband", color='b', linestyle='solid', linewidth=1)
plt.plot(taiex_close['2008-6-1':'2009-5-15'], label="台股大盤", color='black', linewidth=0.8)

plt.axvspan(pd.Timestamp('2008-7-25'), pd.Timestamp('2009-5-5'), color='gray', alpha=0.25)

# 箭頭
plt.annotate('收盤價第一次跌破下軌線', 
             xy=(pd.Timestamp('2008-6-30'), taiex_close.loc['2008-6-30']),  # 箭頭指向的點
             xytext=(pd.Timestamp('2008-6-30'), taiex_close.loc['2008-6-30'] + 500),  # 文字位置往上移
             arrowprops=dict(
                arrowstyle='->',
                connectionstyle='angle,angleA=90,angleB=0',  # 使箭頭垂直
                shrinkA=0,  
                shrinkB=0   
             ),
             fontsize=12, 
             color='red',
             ha='center',  # 水平置中對齊
             va='bottom'   # 垂直對齊在文字底部
)

plt.annotate('收盤價再度跌破下軌線，\n且價格低於第一次跌破價格', 
             xy=(pd.Timestamp('2008-7-25'), taiex_close.loc['2008-7-25']),
             xytext=(pd.Timestamp('2008-7-25'), taiex_close.loc['2008-7-25'] - 2000),
             arrowprops=dict(
                arrowstyle='->',
                connectionstyle='angle,angleA=90,angleB=0',  # 使箭頭垂直
                shrinkA=0,  
                shrinkB=0   
             ),
             fontsize=14, 
             color='red',
             ha='center',
             va='bottom'
)

plt.annotate('收盤價回到均線之上', 
             xy=(pd.Timestamp('2009-5-5'), taiex_close.loc['2009-5-5']),
             xytext=(pd.Timestamp('2009-5-5'), taiex_close.loc['2009-5-5'] + 1000),
             arrowprops=dict(
                arrowstyle='->',
                connectionstyle='angle,angleA=90,angleB=0',  # 使箭頭垂直
                shrinkA=0,  
                shrinkB=0   
             ),
             fontsize=14, 
             color='red',
             ha='center',
             va='bottom'
)

plt.title("台股大盤布林通道（MA300 標準差=2）", fontsize=16)
plt.xlabel("年份", fontsize=14)
plt.xticks(fontsize=14)
plt.ylabel("台股大盤")
plt.yticks(fontsize=14)

plt.legend(fontsize=14)
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()

# fig.savefig('./img/圖 15順勢布林通道濾網示意圖.svg', dpi=300, format='svg', bbox_inches='tight')

### 建立布林通道濾網交易訊號

In [ ]:
bolling_filt = orig_all_cond.copy()

# 將 df 的 index 對齊 us_bollinger_signal_1 的 index，並填充為 True，這樣可以保證日期範圍內的所有日期都包括
aligned_signal = tw_bollinger_signal.reindex(bolling_filt.index, method='ffill', fill_value=True)

# 將 df 中日期對應的 row 設置為 aligned_signal 的值
bolling_filt.loc[aligned_signal.index, :] = aligned_signal.values[:, None]

### 有無濾網_綜合比較

In [ ]:
bollinger_compare_dict = {}

# END_DATE = '2009-3-31'

bollinger_compare_dict['原始條件_無本益比'] = orig_all_cond[START_DATE:END_DATE]
bollinger_compare_dict['原始條件_無本益比_ROE出場條件'] = orig_all_cond[START_DATE:END_DATE] & (roe[START_DATE:END_DATE] > 15)
bollinger_compare_dict['原始條件_無本益比_布林通道'] = orig_all_cond[START_DATE:END_DATE] & bolling_filt[START_DATE:END_DATE]
bollinger_compare_dict['原始條件_無本益比_ROE出場條件_布林通道'] = orig_all_cond[START_DATE:END_DATE] & bolling_filt[START_DATE:END_DATE] & (roe[START_DATE:END_DATE] > 15)

bollinger_compare_dict['原始條件_有本益比'] = orig_all_cond_and_pe_daily[START_DATE:END_DATE]
bollinger_compare_dict['原始條件_有本益比_ROE出場條件'] = (orig_all_cond[START_DATE:END_DATE] & pe_cond_entry_daily[START_DATE:END_DATE]).hold_until((~orig_all_cond[START_DATE:END_DATE]) | pe_cond_exit_daily[START_DATE:END_DATE] | (roe[START_DATE:END_DATE] < 15))
bollinger_compare_dict['原始條件_有本益比_布林通道'] = (orig_all_cond[START_DATE:END_DATE] & pe_cond_entry_daily[START_DATE:END_DATE]).hold_until((~orig_all_cond[START_DATE:END_DATE]) | pe_cond_exit_daily[START_DATE:END_DATE] | ~bolling_filt[START_DATE:END_DATE])
bollinger_compare_dict['原始條件_有本益比_ROE出場條件_布林通道'] = (orig_all_cond[START_DATE:END_DATE] & pe_cond_entry_daily[START_DATE:END_DATE]).hold_until((~orig_all_cond[START_DATE:END_DATE]) | pe_cond_exit_daily[START_DATE:END_DATE] | ~bolling_filt[START_DATE:END_DATE] | (roe[START_DATE:END_DATE] < 15)) 

In [ ]:
bollinger_compare_collecs = sim_conditions(bollinger_compare_dict, resample='M', data=data)

In [ ]:
# bollinger_compare_collecs.plot_creturns()

In [ ]:
bollinger_compare_collecs.plot_strategies_cumm_return().savefig('./img/圖 25 台股策略累積報酬比較圖.svg', format='svg', bbox_inches='tight')

In [ ]:
bollinger_compare_collecs.plot_strategies_MDD()

In [ ]:
bollinger_compare_collecs.selected_stock_count_analysis()

In [ ]:
fig = bollinger_compare_collecs.plot_reps_stock_counts()

fig.savefig('./img/圖 20台股策略入選股數變化比較圖.svg', dpi=300, format='svg', bbox_inches='tight')

#### CAGR比較

In [ ]:
bollinger_compare_df = bollinger_compare_collecs.selected_stock_count_analysis().reset_index()

In [ ]:
# 提取策略前綴，移除 '_布林通道' 作為分組依據
bollinger_compare_df["Base_Strategy"] = bollinger_compare_df["Strategy"].str.replace("_布林通道", "", regex=False)
bollinger_compare_df["Condition"] = bollinger_compare_df["Strategy"].str.contains("_布林通道")

# # 計算有無布林通道的績效差異
# comparison_df = bollinger_compare_df.pivot_table(
#     index="Base_Strategy",  # 使用策略前綴作為索引
#     columns="Condition",
#     values="CAGR (%)"
# )

# # 調整列名，便於圖表呈現
# comparison_df.columns = ["無布林通道", "布林通道"]
# comparison_df = comparison_df.reset_index()

# # 繪製條形圖
# ax = comparison_df.plot(
#     x="Base_Strategy",
#     kind="bar",
#     figsize=(18, 7),
#     xlabel="策略",
#     rot=45,
#     color=['tab:blue', 'orange'],
#     fontsize=14
# )

# plt.title(f"台股 2003~{END_DATE} 布林通道 vs 無布林通道的 CAGR (%) 比較", fontsize=16)
# plt.ylabel("CAGR (%)", fontsize=14)

# # 調整圖例
# plt.legend(["無布林通道", "布林通道"], title="條件", fontsize=12)
# # plt.tight_layout()
# plt.grid(axis='y', alpha=0.6, linestyle='--', linewidth=0.3)
# plt.show()

#### MDD比較

In [ ]:
# # 提取策略前綴，移除 '_布林通道' 作為分組依據
# bollinger_compare_df["Base_Strategy"] = bollinger_compare_df["Strategy"].str.replace("_布林通道", "", regex=False)
# bollinger_compare_df["Condition"] = bollinger_compare_df["Strategy"].str.contains("_布林通道") 

# # 計算有無布林通道的績效差異
# comparison_df = bollinger_compare_df.pivot_table(
#     index="Base_Strategy",  # 使用策略前綴作為索引
#     columns="Condition",
#     values="MDD (%)"
# )

# # 調整列名，便於圖表呈現
# comparison_df.columns = ["無布林通道", "布林通道"]
# comparison_df = comparison_df.reset_index()

# # 繪製條形圖
# ax = comparison_df.plot(
#     x="Base_Strategy",
#     kind="bar",
#     figsize=(18, 7),
#     ylabel="MDD (%)",
#     xlabel="策略",
#     rot=45,
#     color=['tab:blue', 'orange'],
#     fontsize=14
# )


# plt.title(f"台股 2003~{END_DATE} 布林通道 vs 無布林通道的 MDD (%) 比較", fontsize=16)
# plt.ylabel("CAGR (%)", fontsize=14)

# # 調整圖例
# plt.legend(["無布林通道", "布林通道"], title="條件", fontsize=12)
# plt.grid(axis='y', alpha=0.4, linestyle='--')
# plt.show()

#### 合併比較圖表

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 8))

# CAGR
cagr_comparison = bollinger_compare_df.pivot_table(
    index="Base_Strategy",
    columns="Condition",
    values="CAGR (%)"
)
cagr_comparison.columns = ["無布林通道", "布林通道"]
cagr_comparison = cagr_comparison.reset_index()

cagr_comparison.plot(
    x="Base_Strategy",
    kind="bar",
    ax=ax1,
    xlabel="策略",
    rot=45,
    color=['tab:blue', 'orange'],
    fontsize=16,
    legend=False  # 移除個別的legend
)
ax1.set_title(f"台股 2003~{END_DATE} 布林通道 vs 無布林通道的 CAGR (%) 比較", fontsize=16)
ax1.set_ylabel("CAGR (%)", fontsize=12)
ax1.grid(axis='y', alpha=0.4, linestyle='--')

# MDD 
mdd_comparison = bollinger_compare_df.pivot_table(
    index="Base_Strategy",
    columns="Condition",
    values="MDD (%)"
)
mdd_comparison.columns = ["無布林通道", "布林通道"]
mdd_comparison = mdd_comparison.reset_index()

mdd_comparison.plot(
    x="Base_Strategy",
    kind="bar",
    ax=ax2,
    xlabel="策略",
    rot=45,
    color=['tab:blue', 'orange'],
    fontsize=16,
    legend=False
)
ax2.set_title(f"台股 2003~{END_DATE} 布林通道 vs 無布林通道的 MDD (%) 比較", fontsize=16)
ax2.set_ylabel("MDD (%)", fontsize=12)
ax2.grid(axis='y', alpha=0.4, linestyle='--')

# 添加共同的legend在底部中央
handles, labels = ax1.get_legend_handles_labels()
fig.legend(handles, labels, 
          loc='center',
          bbox_to_anchor=(0.5, -0.01),  # 調整legend的位置
          ncol=2,  # 將legend排成兩列
          fontsize=12,
          title_fontsize=12)

plt.tight_layout()
plt.show()

# fig.savefig('./img/圖 18台股2003年至2009年有無加上濾網策略績效比較圖', dpi=300, format='svg', bbox_inches='tight')

In [ ]:
# bollinger_compare_collecs.reports['原始條件_有本益比_布林通道'].plot_strategy_cumm_return(title='原始條件_有本益比_布林通道')

In [ ]:
pe_orig_bollinger_compare_dict = {}

pe_orig_bollinger_compare_dict['原始條件_有本益比'] = orig_all_cond_and_pe_daily[START_DATE:END_DATE]

pe_orig_bollinger_compare_dict['原始條件_有本益比_布林通道'] = (orig_all_cond[START_DATE:END_DATE] & pe_cond_entry_daily[START_DATE:END_DATE]).hold_until((~orig_all_cond[START_DATE:END_DATE]) | pe_cond_exit_daily[START_DATE:END_DATE] | ~bolling_filt[START_DATE:END_DATE])

pe_orig_bollinger_compare_collecs = sim_conditions(pe_orig_bollinger_compare_dict, resample='M', data=data)

pe_orig_bollinger_compare_collecs.plot_strategies_MDD()

In [ ]:
pe_orig_bollinger_compare_collecs.plot_strategies_MDD().savefig('./img/圖 27台股策略MDD比較圖.svg', dpi=300, format='svg', bbox_inches='tight')

---

## 不同換股頻率實驗

In [ ]:
# orig_all_cond_A_rep = backtest.sim(orig_all_cond, resample='YE', data=data)
# orig_all_cond_Q_rep = backtest.sim(orig_all_cond, resample='QE', data=data)
# orig_all_cond_MS_rep = backtest.sim(orig_all_cond, resample='MS', data=data)
# orig_all_cond_M15_rep = backtest.sim(orig_all_cond, resample='MS', resample_offset='15D', data=data)
# orig_all_cond_MS_rep = backtest.sim(orig_all_cond, resample='ME', data=data)
# orig_all_cond_2W_rep = backtest.sim(orig_all_cond, resample='2W', data=data)
# orig_all_cond_W_rep = backtest.sim(orig_all_cond, resample='W', data=data)
# orig_all_cond_D_rep = backtest.sim(orig_all_cond, resample='D', data=data)

### 每季本益比

In [ ]:
# orig_all_cond_pe_A_rep = backtest.sim(orig_all_cond_and_pe, resample='YE', data=data)
# orig_all_cond_pe_AMAR_rep = backtest.sim(orig_all_cond_and_pe, resample='YE', data=data)
# orig_all_cond_pe_Q_rep = backtest.sim(orig_all_cond_and_pe, resample='QE', data=data)
# orig_all_cond_pe_MS_rep = backtest.sim(orig_all_cond_and_pe, resample='MS', data=data)
# orig_all_cond_pe_M15_rep = backtest.sim(orig_all_cond_and_pe, resample='MS', resample_offset='15D', data=data)
# orig_all_cond_pe_M_rep = backtest.sim(orig_all_cond_and_pe, resample='ME', data=data)
# orig_all_cond_pe_2W_rep = backtest.sim(orig_all_cond_and_pe, resample='2W', data=data)
# orig_all_cond_pe_W_rep = backtest.sim(orig_all_cond_and_pe, resample='W', data=data)
# orig_all_cond_pe_D_rep = backtest.sim(orig_all_cond_and_pe, resample='D', data=data)

In [ ]:
# 每年底 resample
pe_cond_entry_A = (pe < 12).resample('A').last()[START_DATE:END_DATE]
pe_cond_exit_A = (pe > 30).resample('A').last()[START_DATE:END_DATE]
all_cond_A = orig_all_cond.resample('A').last()[START_DATE:END_DATE]

all_cond_pe_A = (all_cond_A & pe_cond_entry_A).hold_until(~all_cond_A | pe_cond_exit_A)
all_cond_pe_A_rep = backtest.sim(all_cond_pe_A, resample='A', data=data)

# 每年三月底 resample
pe_cond_entry_AMAR = (pe < 12).resample('A-MAR').last()[START_DATE:END_DATE]
pe_cond_exit_AMAR = (pe > 30).resample('A-MAR').last()[START_DATE:END_DATE]
all_cond_AMAR = orig_all_cond.resample('A-MAR').last()[START_DATE:END_DATE]

all_cond_pe_AMAR = (all_cond_AMAR & pe_cond_entry_AMAR).hold_until(~all_cond_AMAR | pe_cond_exit_AMAR)
all_cond_pe_AMAR_rep = backtest.sim(all_cond_pe_AMAR, resample='A-MAR', data=data)

# 三月底 + 15天
all_cond_pe_AMAR15_rep = backtest.sim(all_cond_pe_AMAR, resample='A-MAR', resample_offset='15D', data=data)

# 每季 resample
pe_cond_entry_Q = (pe < 12).resample('Q').last()[START_DATE:END_DATE]
pe_cond_exit_Q = (pe > 30).resample('Q').last()[START_DATE:END_DATE]
all_cond_Q = orig_all_cond.resample('Q').last()[START_DATE:END_DATE]

all_cond_pe_Q = (all_cond_Q & pe_cond_entry_Q).hold_until(~all_cond_Q| pe_cond_exit_Q )
all_cond_pe_Q_rep = backtest.sim(all_cond_pe_Q, resample='Q', data=data)

# 每月月初 resample
pe_cond_entry_MS = (pe < 12).resample('MS').first()[START_DATE:END_DATE]
pe_cond_exit_MS = (pe > 30).resample('MS').first()[START_DATE:END_DATE]
all_cond_MS = orig_all_cond.resample('MS').first()[START_DATE:END_DATE]

all_cond_pe_MS = (all_cond_MS & pe_cond_entry_MS).hold_until(~all_cond_MS | pe_cond_exit_MS)
all_cond_pe_MS_rep = backtest.sim(all_cond_pe_MS, resample='MS', data=data)

all_cond_pe_M15_rep = backtest.sim(all_cond_pe_MS, resample='M', resample_offset='15D', data=data)

# 每月月底 resample
# pe_cond_entry_M = (pe < 12).resample('ME').last()[START_DATE:END_DATE]
# pe_cond_exit_M = (pe > 30).resample('ME').last()[START_DATE:END_DATE]

all_cond_pe_M = (orig_all_cond & pe_entry).hold_until(~orig_all_cond | pe_exit)
all_cond_pe_M_rep = backtest.sim(all_cond_pe_M, resample='M', data=data)

In [ ]:
# 每兩週 resample

all_cond_pe_2W = (orig_all_cond & pe_entry).hold_until(~orig_all_cond | pe_exit)
all_cond_pe_2W_rep = backtest.sim(all_cond_pe_2W, resample='2W', data=data)
# orig_all_cond_pe_2W_rep.display()

# 每周 resample

all_cond_pe_W = (orig_all_cond & pe_entry).hold_until(~orig_all_cond | pe_exit)
all_cond_pe_W_rep = backtest.sim(all_cond_pe_W, resample='W', data=data)

# 每日 resample

all_cond_pe_D = (orig_all_cond & pe_entry).hold_until(~orig_all_cond | pe_exit)
all_cond_pe_D_rep = backtest.sim(all_cond_pe_D, resample='D', data=data)

In [ ]:
print(f"每年\n{all_cond_pe_A_rep.get_stats()}")
print(f"每年三月底\n{all_cond_pe_AMAR_rep.get_stats()}")
print(f"每年三月底 + 15天\n{all_cond_pe_AMAR15_rep.get_stats()}")
print(f"每季\n{all_cond_pe_Q_rep.get_stats()}")
print(f"每月月初\n{all_cond_pe_MS_rep.get_stats()}")
print(f"每月月中\n{all_cond_pe_M15_rep.get_stats()}")
print(f"每月月底\n{all_cond_pe_M_rep.get_stats()}")
print(f"每兩週\n{all_cond_pe_2W_rep.get_stats()}")
print(f"每周\n{all_cond_pe_W_rep.get_stats()}")
print(f"每日\n{all_cond_pe_D_rep.get_stats()}")

In [ ]:
all_cond_pe_MS_rep.display()

In [ ]:
all_cond_pe_M15_rep.display()

In [ ]:
print(f"每年年底\n入選股數平均：{round(all_cond_pe_A_rep.stock_data['company_count'].mean(), 2)}中位數：{all_cond_pe_A_rep.stock_data['company_count'].median()}, min: {all_cond_pe_A_rep.stock_data['company_count'].min()}, max: {all_cond_pe_A_rep.stock_data['company_count'].max()}")

print(f"每年三月底\n入選股數平均：{round(all_cond_pe_AMAR_rep.stock_data['company_count'].mean(), 2)}中位數：{all_cond_pe_AMAR_rep.stock_data['company_count'].median()}, min: {all_cond_pe_AMAR_rep.stock_data['company_count'].min()}, max: {all_cond_pe_AMAR_rep.stock_data['company_count'].max()}")

print(f"每年三月底 + 15天\n入選股數平均：{round(all_cond_pe_AMAR15_rep.stock_data['company_count'].mean(), 2)}中位數：{all_cond_pe_AMAR15_rep.stock_data['company_count'].median()}, min: {all_cond_pe_AMAR15_rep.stock_data['company_count'].min()}, max: {all_cond_pe_AMAR15_rep.stock_data['company_count'].max()}")

print(f"每季\n入選股數平均：{round(all_cond_pe_Q_rep.stock_data['company_count'].mean(), 2)}中位數：{all_cond_pe_Q_rep.stock_data['company_count'].median()}, min: {all_cond_pe_Q_rep.stock_data['company_count'].min()}, max: {all_cond_pe_Q_rep.stock_data['company_count'].max()}")

print(f"每月月初\n入選股數平均：{round(all_cond_pe_MS_rep.stock_data['company_count'].mean(), 2)}中位數：{all_cond_pe_MS_rep.stock_data['company_count'].median()}, min: {all_cond_pe_MS_rep.stock_data['company_count'].min()}, max: {all_cond_pe_MS_rep.stock_data['company_count'].max()}")

print(f"每月月中\n入選股數平均：{round(all_cond_pe_M15_rep.stock_data['company_count'].mean(), 2)}中位數：{all_cond_pe_M15_rep.stock_data['company_count'].median()}, min: {all_cond_pe_M15_rep.stock_data['company_count'].min()}, max: {all_cond_pe_M15_rep.stock_data['company_count'].max()}")

print(f"每月月底\n入選股數平均：{round(all_cond_pe_M_rep.stock_data['company_count'].mean(), 2)}中位數：{all_cond_pe_M_rep.stock_data['company_count'].median()}, min: {all_cond_pe_M_rep.stock_data['company_count'].min()}, max: {all_cond_pe_M_rep.stock_data['company_count'].max()}")

print(f"每兩週\n入選股數平均：{round(all_cond_pe_2W_rep.stock_data['company_count'].mean(), 2)}中位數：{all_cond_pe_2W_rep.stock_data['company_count'].median()}, min: {all_cond_pe_2W_rep.stock_data['company_count'].min()}, max: {all_cond_pe_2W_rep.stock_data['company_count'].max()}")

print(f"每周\n入選股數平均：{round(all_cond_pe_W_rep.stock_data['company_count'].mean(), 2)}中位數：{all_cond_pe_W_rep.stock_data['company_count'].median()}, min: {all_cond_pe_W_rep.stock_data['company_count'].min()}, max: {all_cond_pe_W_rep.stock_data['company_count'].max()}")

print(f"每日\n入選股數平均：{round(all_cond_pe_D_rep.stock_data['company_count'].mean(), 2)}中位數：{all_cond_pe_D_rep.stock_data['company_count'].median()}, min: {all_cond_pe_D_rep.stock_data['company_count'].min()}, max: {all_cond_pe_D_rep.stock_data['company_count'].max()}")

In [ ]:
data_series = {
    '每年年底': all_cond_pe_A_rep.stock_data['company_count'].dropna(),
    '每年三月月底': all_cond_pe_AMAR_rep.stock_data['company_count'].dropna(),
    # '每年三月月底 + 15天': all_cond_pe_AMAR15_rep.stock_data['company_count'].dropna(),
    '每季': all_cond_pe_Q_rep.stock_data['company_count'].dropna(),
    '月初': all_cond_pe_MS_rep.stock_data['company_count'].dropna(),
    '月中': all_cond_pe_M15_rep.stock_data['company_count'].dropna(),
    '月底': all_cond_pe_M_rep.stock_data['company_count'].dropna(),
    '每兩週': all_cond_pe_2W_rep.stock_data['company_count'].dropna(),
    '每週': all_cond_pe_W_rep.stock_data['company_count'].dropna(),
    '每日': all_cond_pe_D_rep.stock_data['company_count']
}

# 定義不同的 linestyle
linestyles = ['-', '--', '-.', ':', (0, (5, 1)), (0, (3, 1, 1, 1)), (0, (3, 1, 1, 1, 1, 1)), (0, (1, 1)), (0, (5, 2, 1, 2)), (0, (3, 1, 1, 1, 1, 1, 1, 1))]
markers = ['o', 's', 'D', '', '', '', '', '', '', '']

fig = plt.figure(figsize=(20, 8))

for (label, series), linestyle, marker in zip(data_series.items(), linestyles, markers):
    plt.plot(series.index, series.values, linestyle=linestyle, label=label, marker=marker, markersize=2)

# 圖表設定
plt.title('台股_不同換股頻率(resample週期)的入選公司數量變化_每季本益比', fontsize=18)
plt.xlabel('年份', fontsize=14)
plt.ylabel('入選股數', fontsize=16)
plt.legend(fontsize=16)
plt.grid(True, linestyle='--', alpha=0.4)

# 設置 X 軸為每年年份
plt.xticks(
    ticks=pd.date_range(start=series.index.min(), end=series.index.max(), freq="YS"),
    labels=pd.date_range(start=series.index.min(), end=series.index.max(), freq="YS").strftime("%Y"),
    rotation=45,
    fontsize=14
)

# 顯示圖表
plt.show()

fig.savefig('./img/圖 30台股策略使用每季本益比不同換股週期的入選股數變化.svg', dpi=300, format='svg', bbox_inches='tight')

In [ ]:
# all_cond_pe_D_rep.stock_data['company_count'].to_csv('./test_csv_file/resample_D_stock_count.csv')
# all_cond_pe_D_rep.trades.to_csv('./test_csv_file/resample_D_trades.csv')

### reports

In [ ]:
# orig_all_cond_pe_A_rep.display()
# orig_all_cond_pe_Q_rep.display()
# orig_all_cond_pe_MS_rep.display()
# orig_all_cond_pe_M15_rep.display()
# orig_all_cond_pe_M_rep.display()
# orig_all_cond_pe_2W_rep.display()
# orig_all_cond_pe_W_rep.display()
# orig_all_cond_pe_D_rep.display()

### 每日本益比

In [ ]:
# daily_pe_entry_test = (daily_pe < 12)[START_DATE:END_DATE]
# daily_pe_exit_test = (daily_pe > 30)[START_DATE:END_DATE]

In [ ]:
# 每年底 resample
daily_pe_entry_A = (daily_pe < 12).resample('A').last()[START_DATE:END_DATE]
daily_pe_exit_A = (daily_pe > 30).resample('A').last()[START_DATE:END_DATE]

orig_daily_pe_A = (all_cond_A & daily_pe_entry_A).hold_until(~all_cond_A | daily_pe_exit_A)
orig_daily_pe_A_rep = backtest.sim(orig_daily_pe_A, resample='A', data=data)

# 每季 resample
daily_pe_entry_Q = (daily_pe < 12).resample('Q').last()[START_DATE:END_DATE]
daily_pe_exit_Q = (daily_pe > 30).resample('Q').last()[START_DATE:END_DATE]

orig_daily_pe_Q = (all_cond_Q & daily_pe_entry_Q).hold_until(~all_cond_Q | daily_pe_exit_Q)
orig_daily_pe_Q_rep = backtest.sim(orig_daily_pe_Q, resample='Q', data=data)

# 每月月初 resample
daily_pe_entry_MS = (daily_pe < 12).resample('MS').first()[START_DATE:END_DATE]
daily_pe_exit_MS = (daily_pe > 30).resample('MS').first()[START_DATE:END_DATE]

orig_daily_pe_MS = (all_cond_MS & daily_pe_entry_MS).hold_until(~all_cond_MS | daily_pe_exit_MS)
orig_daily_pe_MS_rep = backtest.sim(orig_daily_pe_MS, resample='MS', data=data)

# 每月月中 resample
orig_daily_pe_M15_rep = backtest.sim(orig_daily_pe_MS, resample='MS', resample_offset='15D', data=data)

# 每月月底 resample
orig_daily_pe_M_rep = backtest.sim(orig_all_cond_and_pe_daily, resample='M', data=data)

# 每兩週 resample
daily_pe_entry_2W = (daily_pe < 12).resample('2W').last()[START_DATE:END_DATE]
daily_pe_exit_2W = (daily_pe > 30).resample('2W').last()[START_DATE:END_DATE]

orig_daily_pe_2W = (orig_all_cond & daily_pe_entry_2W).hold_until(~orig_all_cond | daily_pe_exit_2W)
orig_daily_pe_2W_rep = backtest.sim(orig_daily_pe_2W, resample='2W', data=data)

# 每周 resample
daily_pe_entry_W = (daily_pe < 12).resample('W').last()[START_DATE:END_DATE]
daily_pe_exit_W = (daily_pe > 30).resample('W').last()[START_DATE:END_DATE]

orig_daily_pe_W = (orig_all_cond & daily_pe_entry_W).hold_until(~orig_all_cond | daily_pe_exit_W)
orig_daily_pe_W_rep = backtest.sim(orig_daily_pe_W, resample='W', data=data)

In [ ]:
# 每日 resample
orig_daily_pe_D = (orig_all_cond & (daily_pe < 12)).hold_until(~orig_all_cond | (daily_pe > 30))
orig_daily_pe_D_rep = backtest.sim(orig_daily_pe_D, resample='D', data=data)

In [ ]:
# 每三月月底 resample
daily_pe_entry_AMAR = (daily_pe < 12).resample('A-MAR').last()[START_DATE:END_DATE]
daily_pe_exit_AMAR = (daily_pe > 30).resample('A-MAR').last()[START_DATE:END_DATE]
# all_cond_AMAR = orig_all_cond.resample('YE-MAR').last()[START_DATE:END_DATE]

orig_daily_pe_AMAR = (all_cond_AMAR & daily_pe_entry_AMAR).hold_until(~all_cond_AMAR | daily_pe_exit_AMAR)
orig_daily_pe_AMAR_rep = backtest.sim(orig_daily_pe_AMAR, resample='A-MAR', data=data)

orig_daily_pe_AMAR15_rep = backtest.sim(orig_daily_pe_AMAR, resample='A-MAR', resample_offset="15D", data=data)

In [ ]:
# daily_pe_entry_AMAR

In [ ]:
print(f"每年\n{orig_daily_pe_A_rep.get_stats()}")
print(f"每年三月底\n{orig_daily_pe_AMAR_rep.get_stats()}")
print(f"每年三月底 + 15天\n{orig_daily_pe_AMAR15_rep.get_stats()}")
print(f"每季\n{orig_daily_pe_Q_rep.get_stats()}")
print(f"每月月初\n{orig_daily_pe_MS_rep.get_stats()}")
print(f"每月月中\n{orig_daily_pe_M15_rep.get_stats()}")
print(f"每月月底\n{orig_daily_pe_M_rep.get_stats()}")
print(f"每兩週\n{orig_daily_pe_2W_rep.get_stats()}")
print(f"每周\n{orig_daily_pe_W_rep.get_stats()}")
print(f"每日\n{orig_daily_pe_D_rep.get_stats()}")

入選股數統計

In [ ]:
print(f"每年年底\n入選股數平均:{round(orig_daily_pe_A_rep.stock_data['company_count'].dropna().mean(), 2)}, 中位數:{orig_daily_pe_A_rep.stock_data['company_count'].dropna().median()}, min:{orig_daily_pe_A_rep.stock_data['company_count'].dropna().min()}, max:{orig_daily_pe_A_rep.stock_data['company_count'].dropna().max()}")
print(f"每年三月月底\n入選股數平均:{round(orig_daily_pe_AMAR_rep.stock_data['company_count'].dropna().mean(), 2)}, 中位數:{orig_daily_pe_AMAR_rep.stock_data['company_count'].dropna().median()}, min:{orig_daily_pe_AMAR_rep.stock_data['company_count'].dropna().min()}, max:{orig_daily_pe_AMAR_rep.stock_data['company_count'].dropna().max()}")
print(f"每年三月月底 + 15天\n入選股數平均:{round(orig_daily_pe_AMAR15_rep.stock_data['company_count'].dropna().mean(), 2)}, 中位數:{orig_daily_pe_AMAR15_rep.stock_data['company_count'].dropna().median()}, min:{orig_daily_pe_AMAR15_rep.stock_data['company_count'].dropna().min()}, max:{orig_daily_pe_AMAR15_rep.stock_data['company_count'].dropna().max()}")
print(f"每季\n入選股數平均:{round(orig_daily_pe_Q_rep.stock_data['company_count'].dropna().mean(), 2)}, 中位數:{orig_daily_pe_Q_rep.stock_data['company_count'].dropna().median()}, min:{orig_daily_pe_Q_rep.stock_data['company_count'].dropna().min()}, max:{orig_daily_pe_Q_rep.stock_data['company_count'].dropna().max()}")
print(f"每月月初\n入選股數平均:{round(orig_daily_pe_MS_rep.stock_data['company_count'].dropna().mean(), 2)}, 中位數:{orig_daily_pe_MS_rep.stock_data['company_count'].dropna().median()}, min:{orig_daily_pe_MS_rep.stock_data['company_count'].dropna().min()}, max:{orig_daily_pe_MS_rep.stock_data['company_count'].dropna().max()}")
print(f"每月月中\n入選股數平均:{round(orig_daily_pe_M15_rep.stock_data['company_count'].dropna().mean(), 2)}, 中位數:{orig_daily_pe_M15_rep.stock_data['company_count'].dropna().median()}, min:{orig_daily_pe_M15_rep.stock_data['company_count'].dropna().min()}, max:{orig_daily_pe_M15_rep.stock_data['company_count'].dropna().max()}")
print(f"每月月底\n入選股數平均:{round(orig_daily_pe_M_rep.stock_data['company_count'].dropna().mean(), 2)}, 中位數:{orig_daily_pe_M_rep.stock_data['company_count'].dropna().median()}, min:{orig_daily_pe_M_rep.stock_data['company_count'].dropna().min()}, max:{orig_daily_pe_M_rep.stock_data['company_count'].dropna().max()}")
print(f"每兩週\n入選股數平均:{round(orig_daily_pe_2W_rep.stock_data['company_count'].dropna().mean(), 2)}, 中位數:{orig_daily_pe_2W_rep.stock_data['company_count'].dropna().median()}, min:{orig_daily_pe_2W_rep.stock_data['company_count'].dropna().min()}, max:{orig_daily_pe_2W_rep.stock_data['company_count'].dropna().max()}")
print(f"每周\n入選股數平均:{round(orig_daily_pe_W_rep.stock_data['company_count'].dropna().mean(), 2)}, 中位數:{orig_daily_pe_W_rep.stock_data['company_count'].dropna().median()}, min:{orig_daily_pe_W_rep.stock_data['company_count'].dropna().min()}, max:{orig_daily_pe_W_rep.stock_data['company_count'].dropna().max()}")
print(f"每日\n入選股數平均:{round(orig_daily_pe_D_rep.stock_data['company_count'].mean(), 2)}, 中位數:{orig_daily_pe_D_rep.stock_data['company_count'].median()}, min:{orig_daily_pe_D_rep.stock_data['company_count'].min()}, max:{orig_daily_pe_D_rep.stock_data['company_count'].max()}")

In [ ]:
data_series_daily = {
    '每年年底': orig_daily_pe_A_rep.stock_data['company_count'].dropna(),
    # '每年三月月底_QPE': orig_all_cond_pe_AMAR_rep.stock_data['company_count'].dropna(),
    '每年三月月底': orig_daily_pe_AMAR_rep.stock_data['company_count'].dropna(),
    # '每年三月月底 + 15天': orig_daily_pe_AMAR15_rep.stock_data['company_count'].dropna(),
    '每季': orig_daily_pe_Q_rep.stock_data['company_count'].dropna(),
    '月初': orig_daily_pe_MS_rep.stock_data['company_count'].dropna(),
    '月中': orig_daily_pe_M15_rep.stock_data['company_count'].dropna(),
    '月底': orig_daily_pe_M_rep.stock_data['company_count'].dropna(),
    '每兩週': orig_daily_pe_2W_rep.stock_data['company_count'].dropna(),
    '每週': orig_daily_pe_W_rep.stock_data['company_count'].dropna(),
    '每日': orig_daily_pe_D_rep.stock_data['company_count']
}

linestyles = ['-', '--', '-.', ':', (0, (5, 1)), (0, (3, 1, 1, 1)), (0, (3, 1, 1, 1, 1, 1)), (0, (1, 1)), (0, (5, 2, 1, 2)), (0, (3, 1, 1, 1, 1, 1, 1, 1))]
markers = ['', '', '', '', '', '', '', '', '', '']

fig = plt.figure(figsize=(20, 8))

for (label, series), linestyle, marker in zip(data_series_daily.items(), linestyles, markers):
    plt.plot(series.index, series.values, linestyle=linestyle, label=label, marker=marker, markersize=3)

# 圖表設定
plt.title('台股_不同換股頻率(resample週期)的入選公司數量變化_每日本益比', fontsize=18)
plt.xlabel('年份', fontsize=14)
plt.ylabel('入選股數', fontsize=16)
plt.legend(fontsize=16)
plt.grid(True, linestyle='--', alpha=0.4)

# 設置 X 軸為每年年份
plt.xticks(
    ticks=pd.date_range(start=series.index.min(), end=series.index.max(), freq="YS"),
    labels=pd.date_range(start=series.index.min(), end=series.index.max(), freq="YS").strftime("%Y"),
    rotation=45,
    fontsize=14
)

# 顯示圖表
plt.show()

fig.savefig('./img/圖 29台股策略使用每日本益比不同換股週期的入選股數變化.svg', dpi=300, format='svg', bbox_inches='tight')

In [ ]:
data_series_compare = {
    "每季本益比每個月月底resample":all_cond_pe_M_rep.stock_data['company_count'].dropna(), # 每季本益比
    "每日本益比每個月月底resample":orig_daily_pe_M_rep.stock_data['company_count'].dropna(), # 每日本益比
}

fig = plt.figure(figsize=(20, 8))

markers = ['o', 's', 'D', '', '', '', '', '', '', '']

for (label, series), linestyle, marker in zip(data_series_compare.items(), linestyles, markers):
    plt.plot(series.index, series.values, linestyle=linestyle, label=label, marker=marker, markersize=1.5)

# 圖表設定
plt.title('台股_每季本益比 vs 每日本益比_每個月月底resample', fontsize=18)
plt.xlabel('年份', fontsize=14)
plt.ylabel('入選股數', fontsize=16)
plt.legend(fontsize=16)
plt.grid(True, linestyle='--', alpha=0.4)

# 設置 X 軸為每年年份
plt.xticks(
    ticks=pd.date_range(start=series.index.min(), end=series.index.max(), freq="YS"),
    labels=pd.date_range(start=series.index.min(), end=series.index.max(), freq="YS").strftime("%Y"),
    rotation=45,
    fontsize=14
)

# 顯示圖表
plt.show()

fig.savefig('./img/圖 33 台股策略每月月底換股策略入選股數變化比較圖', dpi=300, format='svg', bbox_inches='tight')

In [ ]:
# # 將兩個序列轉換為 DataFrame
# df_compare = pd.DataFrame({
#     'quarterly_pe': all_cond_pe_M_rep.stock_data['company_count'].dropna(),
#     'daily_pe': orig_daily_pe_M_rep.stock_data['company_count'].dropna()
# })

# # 計算差值 (每日本益比 - 每季本益比)
# df_compare['difference'] = df_compare['daily_pe'] - df_compare['quarterly_pe']

# # 計算統計值
# stats = {
#     '最大差距': df_compare['difference'].max(),
#     '最小差距': df_compare['difference'].min(),
#     '平均差距': df_compare['difference'].mean(),
#     '標準差': df_compare['difference'].std()
# }

# # 印出統計結果
# for metric, value in stats.items():
#     print(f"{metric}: {value:.2f}")

# # 找出最大差距和最小差距發生的日期
# max_diff_date = df_compare['difference'].idxmax()
# min_diff_date = df_compare['difference'].idxmin()

# print(f"\n最大差距發生日期: {max_diff_date.strftime('%Y-%m-%d')}")
# print(f"最小差距發生日期: {min_diff_date.strftime('%Y-%m-%d')}")

### reports

In [ ]:
# orig_daily_pe_A_rep.display()
# orig_daily_pe_Q_rep.display()
# orig_daily_pe_MS_rep.display()
# orig_daily_pe_M15_rep.display()
# orig_daily_pe_M_rep.display()
# orig_daily_pe_2W_rep.display()
# orig_daily_pe_W_rep.display()
# orig_daily_pe_D_rep.display()
# orig_daily_pe_AMAR_rep.display()

---

---
## 比較是否考慮某項條件其對績效的影響

### 各因子公司占比

In [ ]:
# 單一條件

all_conditions = {}

all_conditions['ROE五年平均大於15%'] = roe_15[START_DATE:END_DATE]
all_conditions['盈再率小於40%'] = rr_cond[START_DATE:END_DATE]
all_conditions['股利支付率(配息率)三年至少40%'] = payout_ratio_cond[START_DATE:END_DATE]
all_conditions['稅後淨利大於五億新台幣'] = profit_cond[START_DATE:END_DATE]
all_conditions['董監持股比率大於10%'] = hold_cond[START_DATE:END_DATE]
all_conditions['台股上市櫃滿兩年'] = listed[START_DATE:END_DATE]
all_conditions['董監持股比率大於10%_&_台股上市櫃滿兩年'] = (listed & hold_cond)[START_DATE:END_DATE]
all_conditions['本益比小於12進場'] = pe_cond_entry_daily[START_DATE:END_DATE]

all_conditions_count = sim_conditions(all_conditions, resample='M', data=data)

In [ ]:
all_conditions_count.selected_stock_count_analysis(ratio=True)

---

### 排列策略組合

In [ ]:
# 所有條件
condition_dict = {
    'ROE五年平均': roe_15,
    '盈再率': rr_cond,
    '三年配息率': payout_ratio_cond,
    '稅後淨利': profit_cond,
    '董監持股+上市櫃滿兩年': hold_cond & listed # 次要條件
}

# 分離 "董監持股+上市櫃滿兩年"(次要條件) 與其他條件
additional_condition_key = '董監持股+上市櫃滿兩年'
additional_condition = condition_dict.pop(additional_condition_key)

# 剩餘主要條件的 key
keys = list(condition_dict.keys())

# 創建交易訊號字典
signal_dict = {}

# 遍歷 2, 3, 4 的組合長度 (4個主要條件)
for r in range(2, 5):
    combinations = itertools.combinations(keys, r)
    for combo in combinations:
        # 組合名稱
        combo_name = '+'.join(combo)

        # 條件的 AND 結果
        combined_condition = condition_dict[combo[0]]
        for key in combo[1:]:
            combined_condition &= condition_dict[key]

        # 加入不含 "董監持股+上市櫃滿兩年" 的組合
        signal_dict[combo_name] = combined_condition[START_DATE:END_DATE]

        # 加入含 "董監持股+上市櫃滿兩年" 的組合
        combo_name_with_additional = f"{combo_name}_{additional_condition_key}"
        signal_dict[combo_name_with_additional] = (combined_condition & additional_condition)[START_DATE:END_DATE]

# 添加本益比進出場的組合
pe_signal_dict = {}
for name, sig in signal_dict.items():
    # 原組合的本益比進出場
    pe_signal_dict[f"{name}_本益比進出場"] = (sig[START_DATE:END_DATE] & pe_cond_entry_daily[START_DATE:END_DATE]).hold_until((~sig[START_DATE:END_DATE]) | pe_cond_exit_daily[START_DATE:END_DATE])

    # 含 "董監持股+上市櫃滿兩年" 的組合的本益比進出場
    if additional_condition_key in name:
        pe_signal_dict[f"{name}_本益比進出場"] = (sig[START_DATE:END_DATE] & pe_cond_entry_daily[START_DATE:END_DATE]).hold_until((~sig[START_DATE:END_DATE]) | pe_cond_exit_daily[START_DATE:END_DATE])

# 合併所有組合
signal_dict.update(pe_signal_dict)

# print出所有組合名稱
for name, condition in signal_dict.items():
    print(name)

In [ ]:
print("組合條件數量:", len(signal_dict.keys()))

In [ ]:
signal_dict_comb = sim_conditions(signal_dict, resample='M', data=data)
signal_dict_comb.selected_stock_count_analysis()

In [ ]:
signal_dict_comb.selected_stock_count_analysis(ratio=True)

In [ ]:
# signal_dict_comb.selected_stock_count_analysis().to_csv('./performance_file/台股4-2-4-1_table.csv', encoding='cp950')

### 綜合比較

In [ ]:
pd.options.display.float_format = '{:.2f}'.format


def analyze_signal_groups(signal_dict_comb):
    # 初始化結果字典
    results = {
        '包含ROE五年平均': [],
        '包含盈再率': [],
        '包含配息率': [],
        '包含稅後淨利': [],
        '包含董監持股+上市櫃滿兩年': []
    }

    # 遍歷策略並分類
    for strategy in signal_dict_comb.index:
        if 'ROE五年平均' in strategy:
            results['包含ROE五年平均'].append(strategy)
        if '盈再率' in strategy:
            results['包含盈再率'].append(strategy)
        if '三年配息率' in strategy:
            results['包含配息率'].append(strategy)
        if '稅後淨利' in strategy:
            results['包含稅後淨利'].append(strategy)
        if '董監持股+上市櫃滿兩年' in strategy:
            results['包含董監持股+上市櫃滿兩年'].append(strategy)

    # 計算每組的平均值，細分為包含/不包含條件、有/無本益比進出場、綜合
    summary_data = []
    for group, strategies in results.items():
        condition_name = group.split('包含')[1]

        # 包含條件與不包含條件
        for include_condition in [True, False]:
            filtered_strategies_condition = [s for s in signal_dict_comb.index if (condition_name in s) == include_condition]

            # 細分有/無本益比進出場
            for include_pe in [True, False]:
                filtered_strategies = [s for s in filtered_strategies_condition if ('本益比進出場' in s) == include_pe]
                if filtered_strategies:
                    filtered_df = signal_dict_comb.loc[filtered_strategies]
                    avg_cagr = filtered_df['CAGR (%)'].mean()
                    avg_mdd = filtered_df['MDD (%)'].mean()
                    avg_selected_stock = filtered_df['入選股數平均'].mean()
                    condition_label = f"{'包含' if include_condition else '不包含'}{condition_name} ({'含本益比進出場' if include_pe else '不含本益比進出場'})"
                    summary_data.append([condition_label, avg_cagr, avg_mdd, avg_selected_stock])

            # 綜合分析
            combined_df = signal_dict_comb.loc[filtered_strategies_condition]
            if not combined_df.empty:
                avg_cagr_combined = combined_df['CAGR (%)'].mean()
                avg_mdd_combined = combined_df['MDD (%)'].mean()
                avg_selected_stock_combined = combined_df['入選股數平均'].mean()
                condition_label = f"{'包含' if include_condition else '不包含'}{condition_name} (綜合)"
                summary_data.append([condition_label, avg_cagr_combined, avg_mdd_combined, avg_selected_stock_combined])

    # 返回新的 DataFrame
    summary_df = pd.DataFrame(summary_data, columns=['Group', 'Average CAGR (%)', 'Average MDD (%)', 'Average Selected Stock Count'])
    return summary_df

# signal_dict_comb = sim_conditions(signal_dict, resample='ME', data=data)
summary_df = analyze_signal_groups(signal_dict_comb.selected_stock_count_analysis())
summary_df

### 繪圖

In [ ]:
# 創建DataFrame
df = summary_df.copy()

# # 處理數據：提取所需的行（包含/不包含且是含本益比進出場的組合）
# mask = df['Group'].str.contains('含本益比進出場')
# filtered_df = df[mask].copy()

# # 定義類別順序
# categories = ['ROE五年平均', '盈再率', '配息率', '稅後淨利']

# # 創建類別對應的數據
# included_values = []
# excluded_values = []

# for category in categories:
#     # 獲取包含該類別的資料
#     included_mask = filtered_df['Group'].str.contains(f'包含{category}')
#     excluded_mask = filtered_df['Group'].str.contains(f'不包含{category}')
    
#     included_value = filtered_df[included_mask]['Average CAGR (%)'].values[0]
#     excluded_value = filtered_df[excluded_mask]['Average CAGR (%)'].values[0]
    
#     included_values.append(included_value)
#     excluded_values.append(excluded_value)

# # 設置圖表風格和大小
# # plt.style.use('seaborn')
# plt.figure(figsize=(14, 6))

# # 設置bar的寬度
# width = 0.17

# # 設置x軸位置
# x = np.arange(len(categories))

# # 繪製條形圖
# bars1 = plt.bar(x - width/2, included_values, width, label='包含', color='tab:blue')
# bars2 = plt.bar(x + width/2, excluded_values, width, label='不包含', color='orange')

# # 設置圖表標題和標籤
# plt.title('台股 有無選股原則中其中一項因子的組合 CAGR 平均 (含本益比進出場)', fontsize=12)
# plt.xlabel('指標類別')
# plt.ylabel('Average CAGR (%)')

# # 設置x軸刻度和標籤
# plt.xticks(x, categories)

# # 設置y軸範圍
# plt.ylim(0, 16)

# # 添加網格線
# plt.grid(True, linestyle='--', alpha=0.8)

# benchmark_value = signal_dict_comb.selected_stock_count_analysis().loc['ROE五年平均+盈再率+三年配息率+稅後淨利_董監持股+上市櫃滿兩年_本益比進出場']['CAGR (%)']
# plt.axhline(y=benchmark_value, color='red', linestyle='--', label='所有條件+本益比進出場 CAGR', linewidth=0.3)

# # 添加圖例
# plt.legend(loc='upper center', bbox_to_anchor=(1.15, 1), fontsize=10)

# # 調整布局
# plt.tight_layout()

# # 顯示圖表
# plt.show()

# 定義類別順序
categories = ['ROE五年平均', '盈再率', '配息率', '稅後淨利']

# 創建類別對應的數據
included_no_pe_values = []  # 包含但不含本益比
included_with_pe_values = []  # 包含且含本益比
excluded_no_pe_values = []  # 不包含且不含本益比
excluded_with_pe_values = []  # 不包含且含本益比

for category in categories:
    # 包含且不含本益比
    mask_included_no_pe = df['Group'].str.contains(f'包含{category} \(不含本益比進出場\)')
    included_no_pe_values.append(df[mask_included_no_pe]['Average CAGR (%)'].values[0])
    
    # 包含且含本益比
    mask_included_with_pe = df['Group'].str.contains(f'包含{category} \(含本益比進出場\)')
    included_with_pe_values.append(df[mask_included_with_pe]['Average CAGR (%)'].values[0])
    
    # 不包含且不含本益比
    mask_excluded_no_pe = df['Group'].str.contains(f'不包含{category} \(不含本益比進出場\)')
    excluded_no_pe_values.append(df[mask_excluded_no_pe]['Average CAGR (%)'].values[0])
    
    # 不包含且含本益比
    mask_excluded_with_pe = df['Group'].str.contains(f'不包含{category} \(含本益比進出場\)')
    excluded_with_pe_values.append(df[mask_excluded_with_pe]['Average CAGR (%)'].values[0])

# 設置圖表大小
fig = plt.figure(figsize=(10, 6))

# 設置bar的寬度
width = 0.17

# 設置x軸位置
x = np.arange(len(categories))

# 繪製條形圖
bars1 = plt.bar(x - width*1.5, included_no_pe_values, width, 
                label='包含某項因子_無本益比進出場', color='lightblue')
bars2 = plt.bar(x - width/2, included_with_pe_values, width, 
                label='包含某項因子_加上本益比進出場', color='tab:blue')
bars3 = plt.bar(x + width/2, excluded_no_pe_values, width, 
                label='不包含某項因子_不含本益比', color='bisque')
bars4 = plt.bar(x + width*1.5, excluded_with_pe_values, width, 
                label='不包含某項因子_加上本益比進出場', color='orange')

# 設置圖表標題和標籤
plt.title(f'台股 有無選股原則中其中一項因子的組合 CAGR 平均 ({START_DATE}-{END_DATE} resample="M")', fontsize=12)
plt.xlabel('指標類別')
plt.ylabel('平均 CAGR (%)', fontsize=12)

# 設置x軸刻度和標籤
plt.xticks(x, categories, fontsize=12)

# 設置y軸範圍
plt.ylim(0, 18)

# # 添加網格線
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.grid(axis='x', linestyle='--', alpha=0.8)

# 添加基準線
benchmark_value = signal_dict_comb.selected_stock_count_analysis().loc['ROE五年平均+盈再率+三年配息率+稅後淨利_董監持股+上市櫃滿兩年_本益比進出場']['CAGR (%)']
plt.axhline(y=benchmark_value, color='red', linestyle='--', 
            label='所有條件+本益比進出場 CAGR', linewidth=0.4)

# # 在bar上添加數值標籤
# def add_value_labels(bars):
#     for bar in bars:
#         height = bar.get_height()
#         plt.text(bar.get_x() + bar.get_width()/2., height,
#                 f'{height:.2f}%',
#                 ha='center', va='bottom',
#                 fontsize=8)

# add_value_labels(bars1)
# add_value_labels(bars2)
# add_value_labels(bars3)
# add_value_labels(bars4)

# 添加圖例
plt.legend()
# plt.legend(loc='upper center', bbox_to_anchor=(1, 1), fontsize=12)

# 調整布局
plt.tight_layout()

# 顯示圖表
plt.show()

fig.savefig('./img/圖 37台股有無其中一項因子的策略組合之CAGR平均.svg', format='svg', bbox_inches='tight')

In [ ]:
# 創建DataFrame
df = summary_df.copy()


# 定義類別順序
categories = ['ROE五年平均', '盈再率', '配息率', '稅後淨利']

# 創建類別對應的數據
included_no_pe_values = []  # 包含但不含本益比
included_with_pe_values = []  # 包含且含本益比
excluded_no_pe_values = []  # 不包含且不含本益比
excluded_with_pe_values = []  # 不包含且含本益比

for category in categories:
    # 包含且不含本益比
    mask_included_no_pe = df['Group'].str.contains(f'包含{category} \(不含本益比進出場\)')
    included_no_pe_values.append(df[mask_included_no_pe]['Average Selected Stock Count'].values[0])
    
    # 包含且含本益比
    mask_included_with_pe = df['Group'].str.contains(f'包含{category} \(含本益比進出場\)')
    included_with_pe_values.append(df[mask_included_with_pe]['Average Selected Stock Count'].values[0])
    
    # 不包含且不含本益比
    mask_excluded_no_pe = df['Group'].str.contains(f'不包含{category} \(不含本益比進出場\)')
    excluded_no_pe_values.append(df[mask_excluded_no_pe]['Average Selected Stock Count'].values[0])
    
    # 不包含且含本益比
    mask_excluded_with_pe = df['Group'].str.contains(f'不包含{category} \(含本益比進出場\)')
    excluded_with_pe_values.append(df[mask_excluded_with_pe]['Average Selected Stock Count'].values[0])

# 設置圖表大小
fig = plt.figure(figsize=(14, 6))

# 設置bar的寬度
width = 0.17

# 設置x軸位置
x = np.arange(len(categories))

# 繪製條形圖
bars1 = plt.bar(x - width*1.5, included_no_pe_values, width, 
                label='包含某項因子_無本益比進出場', color='lightblue')
bars2 = plt.bar(x - width/2, included_with_pe_values, width, 
                label='包含某項因子_加上本益比進出場', color='tab:blue')
bars3 = plt.bar(x + width/2, excluded_no_pe_values, width, 
                label='不包含某項因子_不含本益比', color='bisque')
bars4 = plt.bar(x + width*1.5, excluded_with_pe_values, width, 
                label='不包含某項因子_含本益比', color='orange')

# 設置圖表標題和標籤
plt.title('台股 有無選股原則中其中一項因子的組合 入選股數 平均 (2003-2024 resample="M")', fontsize=16)
plt.xlabel('指標類別')
plt.ylabel('平均入選股數', fontsize=14)

# 設置x軸刻度和標籤
plt.xticks(x, categories, fontsize=14)

# # 設置y軸範圍
# plt.ylim(0, 16)

# # 添加網格線
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.grid(axis='x', linestyle='--', alpha=0.8)



# 添加圖例
plt.legend(loc='upper center', ncol=2, fontsize=14)

# 調整布局
plt.tight_layout()

# 顯示圖表
plt.show()

fig.savefig('./img/圖 38台股有無其中一項因子的策略組合之平均入選股數.svg', format='svg', bbox_inches='tight')

In [ ]:
# # 創建DataFrame
# df = summary_df

# # 設定中文字型
# plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei']
# plt.rcParams['axes.unicode_minus'] = False

# # 處理數據：提取所需的行（包含/不包含且是含本益比進出場的組合）
# mask = df['Group'].str.contains('不含本益比進出場')
# filtered_df = df[mask].copy()

# # 定義類別順序
# categories = ['ROE五年平均', '盈再率', '配息率', '稅後淨利']

# # 創建類別對應的數據
# included_values = []
# excluded_values = []

# for category in categories:
#     # 獲取包含該類別的資料
#     included_mask = filtered_df['Group'].str.contains(f'包含{category}')
#     excluded_mask = filtered_df['Group'].str.contains(f'不包含{category}')
    
#     included_value = filtered_df[included_mask]['Average CAGR (%)'].values[0]
#     excluded_value = filtered_df[excluded_mask]['Average CAGR (%)'].values[0]
    
#     included_values.append(included_value)
#     excluded_values.append(excluded_value)

# # 設置圖表風格和大小
# # plt.style.use('seaborn')
# plt.figure(figsize=(14, 5))

# # 設置bar的寬度
# width = 0.17

# # 設置x軸位置
# x = np.arange(len(categories))

# # 繪製條形圖
# bars1 = plt.bar(x - width/2, included_values, width, label='包含', color='tab:blue')
# bars2 = plt.bar(x + width/2, excluded_values, width, label='不包含', color='orange')

# # 設置圖表標題和標籤
# plt.title('台股 有無選股原則中其中一項因子的組合 CAGR 平均 (不含本益比進出場)', fontsize=12)
# plt.xlabel('指標類別')
# plt.ylabel('Average CAGR (%)')

# # 設置x軸刻度和標籤
# plt.xticks(x, categories)



# # 設置y軸範圍
# plt.ylim(0, 16)

# # 添加網格線
# plt.grid(True, linestyle='--', alpha=0.8)

# benchmark_value=signal_dict_comb.selected_stock_count_analysis().loc['ROE五年平均+盈再率+三年配息率+稅後淨利_董監持股+上市櫃滿兩年']['CAGR (%)']
# plt.axhline(y=benchmark_value, color='red', linestyle='--', label='所有條件_沒有本益比進出場CAGR', linewidth=0.3)

# # 添加圖例
# plt.legend(loc='upper center', bbox_to_anchor=(1.15, 1), fontsize=12)


# # 調整布局
# plt.tight_layout()

# # 顯示圖表
# plt.show()

---

In [ ]:
sum_compare_dict={}

sig_1 = rr_cond & payout_ratio_cond
sig_2 = payout_ratio_cond & profit_cond
sig_3 = rr_cond & profit_cond

sig_4 = rr_cond & payout_ratio_cond & profit_cond
sig_5 = rr_cond & payout_ratio_cond & profit_cond & roe_15

sig_filter = hold_cond & listed

sum_compare_dict['盈再率+配息率'] = sig_1[START_DATE:END_DATE]
sum_compare_dict['盈再率+配息率_本益比'] = (sig_1 & pe_cond_entry_daily).hold_until((~sig_1) | pe_cond_exit_daily)[START_DATE:END_DATE]
sum_compare_dict['盈再率+配息率+董監持股+上市櫃滿兩年'] = (sig_1 & sig_filter)[START_DATE:END_DATE]
sum_compare_dict['盈再率+配息率+董監持股+上市櫃滿兩年_本益比'] = (sig_1 & sig_filter & pe_cond_entry_daily).hold_until((~(sig_1 & sig_filter)) | pe_cond_exit_daily)[START_DATE:END_DATE]

sum_compare_dict['配息率+稅後淨利'] = sig_2[START_DATE:END_DATE]
sum_compare_dict['配息率+稅後淨利_本益比'] = (sig_2 & pe_cond_entry_daily).hold_until((~sig_2) | pe_cond_exit_daily)[START_DATE:END_DATE]
sum_compare_dict['配息率+稅後淨利+董監持股+上市櫃滿兩年'] = (sig_2 & sig_filter)[START_DATE:END_DATE]
sum_compare_dict['配息率+稅後淨利+董監持股+上市櫃滿兩年_本益比'] = (sig_2 & sig_filter & pe_cond_entry_daily).hold_until((~(sig_2 & sig_filter)) | pe_cond_exit_daily)[START_DATE:END_DATE]

sum_compare_dict['盈再率+稅後淨利'] = sig_3[START_DATE:END_DATE]
sum_compare_dict['盈再率+稅後淨利_本益比'] = (sig_3 & pe_cond_entry_daily).hold_until((~sig_3) | pe_cond_exit_daily)[START_DATE:END_DATE]
sum_compare_dict['盈再率+稅後淨利+董監持股+上市櫃滿兩年'] = (sig_3 & sig_filter)[START_DATE:END_DATE]
sum_compare_dict['盈再率+稅後淨利+董監持股+上市櫃滿兩年_本益比'] = (sig_3 & sig_filter & pe_cond_entry_daily).hold_until((~(sig_3 & sig_filter)) | pe_cond_exit_daily)[START_DATE:END_DATE]

sum_compare_dict['盈再率+配息率+稅後淨利'] = sig_4[START_DATE:END_DATE]
sum_compare_dict['盈再率+配息率+稅後淨利_本益比'] = (sig_4 & pe_cond_entry_daily).hold_until((~sig_4) | pe_cond_exit_daily)[START_DATE:END_DATE]
sum_compare_dict['盈再率+配息率+稅後淨利+董監持股+上市櫃滿兩年'] = (sig_4 & sig_filter)[START_DATE:END_DATE]
sum_compare_dict['盈再率+配息率+稅後淨利+董監持股+上市櫃滿兩年_本益比'] = (sig_4 & sig_filter & pe_cond_entry_daily).hold_until((~(sig_4 & sig_filter)) | pe_cond_exit_daily)[START_DATE:END_DATE]

sum_compare_dict['ROE五年平均+盈再率+配息率+稅後淨利'] = sig_5[START_DATE:END_DATE]
sum_compare_dict['ROE五年平均+盈再率+配息率+稅後淨利_本益比'] = (sig_5 & pe_cond_entry_daily).hold_until((~sig_5) | pe_cond_exit_daily)[START_DATE:END_DATE]
sum_compare_dict['ROE五年平均+盈再率+配息率+稅後淨利+董監持股+上市櫃滿兩年'] = (sig_5 & sig_filter)[START_DATE:END_DATE]
sum_compare_dict['ROE五年平均+盈再率+配息率+稅後淨利+董監持股+上市櫃滿兩年_本益比'] = (sig_5 & sig_filter & pe_cond_entry_daily).hold_until((~(sig_5 & sig_filter)) | pe_cond_exit_daily)[START_DATE:END_DATE]

sum_compare_strat_collecs = sim_conditions(sum_compare_dict, resample='M', data=data) 

sum_compare_strat_collecs.selected_stock_count_analysis()

In [ ]:
sum_compare_strat_collecs.selected_stock_count_analysis(ratio=True)

In [ ]:
sum_compare_strat_collecs.reports['盈再率+配息率+董監持股+上市櫃滿兩年_本益比'].display()

In [ ]:
df = sum_compare_strat_collecs.selected_stock_count_analysis()
df.reset_index(inplace=True)

In [ ]:
# Clean any leading/trailing spaces in the 'Strategy' column
df['Strategy'] = df['Strategy'].str.strip()

# Extract prefixes and suffixes for categorization
# Ensure the entire strategy is matched correctly
prefix_pattern = r'^(盈再率\+配息率\+稅後淨利|盈再率\+配息率|配息率\+稅後淨利|ROE五年平均\+盈再率\+配息率\+稅後淨利)'
suffix_pattern = r'(\+董監持股\+上市櫃滿兩年_本益比|\+董監持股\+上市櫃滿兩年|_本益比)$'
df['Prefix'] = df['Strategy'].str.extract(prefix_pattern)
df['Suffix'] = df['Strategy'].str.extract(suffix_pattern)

# Add a category for strategies without any suffix
df['Suffix'] = df['Suffix'].fillna('不含_董監持股+上市櫃滿兩年_本益比進出場')

# Ensure all combinations of Prefix and Suffix are present
prefixes = ['盈再率+配息率', '配息率+稅後淨利', '盈再率+配息率+稅後淨利', 'ROE五年平均+盈再率+配息率+稅後淨利']
suffixes = ['不含_董監持股+上市櫃滿兩年_本益比進出場', '_本益比', '+董監持股+上市櫃滿兩年', '+董監持股+上市櫃滿兩年_本益比']
all_combinations = pd.MultiIndex.from_product([prefixes, suffixes], names=['Prefix', 'Suffix'])

# Reindex to ensure all combinations exist and fill missing values with NaN
grouped_df = df.groupby(['Prefix', 'Suffix'])['CAGR (%)'].mean().reindex(all_combinations, fill_value=np.nan).unstack()

# Reorder grouped_df columns and index to match the desired order
grouped_df = grouped_df.loc[prefixes, suffixes]

# Validate grouped_df for missing data
print("Grouped DataFrame:")
print(grouped_df)

# Define bar positions and width
x = np.arange(len(grouped_df))  # Number of prefix groups
width = 0.15  # Width of each bar

# Define suffix colors
suffix_colors = {
    '不含_董監持股+上市櫃滿兩年_本益比進出場': 'tab:gray',
    '_本益比': 'orange',
    '+董監持股+上市櫃滿兩年': 'tab:blue',
    '+董監持股+上市櫃滿兩年_本益比': 'tab:red'
}

# Create the plot
fig, ax = plt.subplots(figsize=(18, 8))

# Plot each suffix category
for i, suffix in enumerate(grouped_df.columns):
    ax.bar(
        x + i * width,
        grouped_df[suffix].fillna(0),  # Replace NaN with 0 for plotting
        width=width,
        label=suffix.replace('_本益比', '+本益比進出場'),  # Escape underscores for legend display
        color=suffix_colors.get(suffix, 'gray')
    )

# Add labels, title, and legend
ax.set_xticks(x + width * (len(grouped_df.columns) - 1) / 2)
ax.set_xticklabels(grouped_df.index, rotation=0, ha='center', fontsize=18)
ax.set_ylabel('CAGR (%)', fontsize=18)
ax.set_yticklabels([f'{tick:.0f}' for tick in ax.get_yticks()], fontsize=18)

ax.set_title('台股2003/3/31-2024/12/31 比較是否考慮某項條件其對績效的影響 CAGR (%) 比較', fontsize=18)
ax.legend(loc='upper right', ncol=2, fontsize=14)

# Show the plot
plt.tight_layout()
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

fig.savefig('./img/圖 39台股策略組合績效圖.svg', format='svg', bbox_inches='tight')

In [ ]:
# Clean any leading/trailing spaces in the 'Strategy' column
df['Strategy'] = df['Strategy'].str.strip()

# Extract prefixes and suffixes for categorization
# Ensure the entire strategy is matched correctly
prefix_pattern = r'^(盈再率\+配息率\+稅後淨利|盈再率\+配息率|配息率\+稅後淨利|ROE五年平均\+盈再率\+配息率\+稅後淨利)'
suffix_pattern = r'(\+董監持股\+上市櫃滿兩年_本益比|\+董監持股\+上市櫃滿兩年|_本益比)$'
df['Prefix'] = df['Strategy'].str.extract(prefix_pattern)
df['Suffix'] = df['Strategy'].str.extract(suffix_pattern)

# Add a category for strategies without any suffix
df['Suffix'] = df['Suffix'].fillna('不含董監持股+上市櫃滿兩年&本益比進出場')

# Ensure all combinations of Prefix and Suffix are present
prefixes = ['盈再率+配息率', '配息率+稅後淨利', '盈再率+配息率+稅後淨利', 'ROE五年平均+盈再率+配息率+稅後淨利']
suffixes = ['不含董監持股+上市櫃滿兩年&本益比進出場', '_本益比', '+董監持股+上市櫃滿兩年', '+董監持股+上市櫃滿兩年_本益比']
all_combinations = pd.MultiIndex.from_product([prefixes, suffixes], names=['Prefix', 'Suffix'])

# Reindex to ensure all combinations exist and fill missing values with NaN
grouped_df = df.groupby(['Prefix', 'Suffix'])['入選股數平均'].mean().reindex(all_combinations, fill_value=np.nan).unstack()

# Reorder grouped_df columns and index to match the desired order
grouped_df = grouped_df.loc[prefixes, suffixes]

# Validate grouped_df for missing data
print("Grouped DataFrame:")
print(grouped_df)

# Define bar positions and width
x = np.arange(len(grouped_df))  # Number of prefix groups
width = 0.15  # Width of each bar

# Define suffix colors
suffix_colors = {
    '不含董監持股+上市櫃滿兩年&本益比進出場': 'tab:gray',
    '_本益比': 'orange',
    '+董監持股+上市櫃滿兩年': 'tab:blue',
    '+董監持股+上市櫃滿兩年_本益比': 'tab:red'
}

# Create the plot
fig, ax = plt.subplots(figsize=(18, 8))

# Plot each suffix category
for i, suffix in enumerate(grouped_df.columns):
    ax.bar(
        x + i * width,
        grouped_df[suffix].fillna(0),  # Replace NaN with 0 for plotting
        width=width,
        label=suffix.replace('_本益比', '+本益比進出場'),  # Escape underscores for legend display
        color=suffix_colors.get(suffix, 'gray')
    )

# Add labels, title, and legend
ax.set_xticks(x + width * (len(grouped_df.columns) - 1) / 2)
ax.set_xticklabels(grouped_df.index, rotation=0, ha='center', fontsize=18)
ax.set_ylabel('入選股數平均數', fontsize=18)
ax.set_yticklabels([f'{tick:.0f}' for tick in ax.get_yticks()], fontsize=18)
ax.set_title('台股2003/3/31-2024/9/30 比較是否考慮某項條件其對績效的影響 入選股數平均 比較', fontsize=16)
ax.legend(loc='upper center', ncol=2, fontsize=18)

# Show the plot
plt.tight_layout()
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

fig.savefig('./img/圖 40台股策略組合平均入選股數比較圖.svg', format='svg', bbox_inches='tight')

---

## 調整財務比率篩選標準

### 進場條件
- 本益比進出場

In [ ]:
# orig_all_opt_pe_roe = (orig_all_cond & pe_cond_entry_daily[START_DATE:END_DATE]).hold_until((~orig_all_cond[START_DATE:END_DATE]) | pe_cond_exit_daily[START_DATE:END_DATE] | (roe[START_DATE:END_DATE] < 15))

pe_entry_test_dic = {}

for p in range(8, 15, 2):
    pe_entry_test = daily_pe.resample('M').last() < p
    pe_entry_test_dic[f'原始條件_本益比小於{p}進場'] = (orig_all_cond & pe_entry_test[START_DATE:END_DATE]).hold_until((~orig_all_cond[START_DATE:END_DATE]) | pe_cond_exit_daily[START_DATE:END_DATE])

pe_entry_test_comb = sim_conditions(pe_entry_test_dic, resample="M", data=data)
pe_entry_test_comb.selected_stock_count_analysis()

### ROE

In [ ]:
roe_value_cond = {}
no_roe_conds = (rr_cond & profit_cond & hold_cond & listed & payout_ratio_cond)[START_DATE:END_DATE]

for i in range(10, 31, 5): # 大於 10~25%
    for n in range(3, 6): # 3, 4, 5年平均
        
        roe_rolling = roe.rolling(n).mean()
        roe_cond_opt = (roe_rolling > i)[START_DATE:END_DATE]

        roe_value_cond[f'roe_{n}y_{i}_無本益比進出場'] = (roe_cond_opt & no_roe_conds)[START_DATE:END_DATE]
        roe_value_cond[f'roe_{n}y_{i}_有本益比進出場'] = ((roe_cond_opt &  no_roe_conds & pe_cond_entry_daily[START_DATE:END_DATE]).hold_until((~(roe_cond_opt &  no_roe_conds)) | pe_cond_exit_daily[START_DATE:END_DATE]))
        
roe_collection = sim_conditions(roe_value_cond, resample='M', data=data)
roe_collection.selected_stock_count_analysis()

In [ ]:
roe_collection.plot_reps_stock_counts(['roe_4y_15_有本益比進出場', 'roe_5y_15_有本益比進出場'])

In [ ]:
# roe_collection.reports['roe_5y_10_有本益比進出場'].display()

In [ ]:
roe_collection_df = roe_collection.selected_stock_count_analysis()
roe_collection_df.reset_index(inplace=True)

In [ ]:
# 建立顏色映射
colors = {
    '10': 'tab:blue',  # 藍色
    '15': 'orange',  # 綠色
    '20': 'tab:green',  # 橘色
    '25': 'tab:red'   # 紅色
}

# 提取年份和ROE閾值
roe_collection_df['Year'] = roe_collection_df['Strategy'].str.extract(r'roe_(\d)y')
roe_collection_df['ROE'] = roe_collection_df['Strategy'].str.extract(r'_(\d+)_')
roe_collection_df['PE'] = roe_collection_df['Strategy'].str.contains('有本益比')

# 創建子圖，設定比例為1:2，並留出下方空間給legend
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

# 獲取唯一的年份值
years = sorted(roe_collection_df['Year'].unique())

# 設定長條的寬度
bar_width = 0.15

# 計算每個年份組的位置
positions = np.arange(len(years))

# 儲存所有CAGR值用於設定y軸範圍
all_cagr_values = []

# 繪製有本益比的長條圖
bars = []  # 儲存長條物件用於之後設定legend
for i, roe_value in enumerate(['10', '15', '20', '25']):
    mask_with_pe = (roe_collection_df['ROE'] == roe_value) & (roe_collection_df['PE'])
    data_with_pe = [roe_collection_df[mask_with_pe & (roe_collection_df['Year'] == year)]['CAGR (%)'].values[0] 
                    if len(roe_collection_df[mask_with_pe & (roe_collection_df['Year'] == year)]) > 0 else 0 
                    for year in years]
    all_cagr_values.extend(data_with_pe)
    
    # 計算長條的位置
    x = positions + (i - 1.5) * bar_width

    # 繪製長條
    bar = ax1.bar(x, data_with_pe, bar_width, label=f'ROE {roe_value}%', color=colors[roe_value])
    bars.append(bar)

# 繪製無本益比的長條圖
for i, roe_value in enumerate(['10', '15', '20', '25']):
    mask_without_pe = (roe_collection_df['ROE'] == roe_value) & (~roe_collection_df['PE'])
    data_without_pe = [roe_collection_df[mask_without_pe & (roe_collection_df['Year'] == year)]['CAGR (%)'].values[0] 
                      if len(roe_collection_df[mask_without_pe & (roe_collection_df['Year'] == year)]) > 0 else 0 
                      for year in years]
    all_cagr_values.extend(data_without_pe)
    
    # 計算長條的位置
    x = positions + (i - 1.5) * bar_width

    # 繪製長條
    ax2.bar(x, data_without_pe, bar_width, color=colors[roe_value])

# 計算y軸範圍
y_min = min(all_cagr_values)
y_max = max(all_cagr_values)
y_margin = (y_max - y_min) * 0.1

# 設定兩個子圖的共同y軸範圍
ax1.set_ylim(y_min - y_margin, y_max + y_margin)
ax2.set_ylim(y_min - y_margin, y_max + y_margin)

# 設定左方子圖（有本益比）
ax1.set_xticks(positions)
ax1.set_xticklabels([f'ROE{year}年平均' for year in years])
ax1.set_title('有本益比進出場策略的CAGR比較', fontsize=16)
ax1.set_xlabel('年份', fontsize=12)
ax1.set_ylabel('CAGR (%)', fontsize=12)

# 設定右方子圖（無本益比）
ax2.set_xticks(positions)
ax2.set_xticklabels([f'ROE{year}年平均' for year in years])
ax2.set_title('無本益比進出場策略的CAGR比較', fontsize=16)
ax2.set_xlabel('年份', fontsize=12)
ax2.set_ylabel('CAGR (%)', fontsize=12)

# 調整版面配置
plt.tight_layout()

ax1.grid(axis='y', linestyle='--', alpha=0.7)
ax2.grid(axis='y', linestyle='--', alpha=0.7)

# 在兩圖中間下方添加legend
legend = fig.legend(bars, [f'ROE N 年平均 > {roe_value}%' for roe_value in ['10', '15', '20', '25']], 
                   loc='center', bbox_to_anchor=(0.5, 0.02),
                   ncol=4, frameon=False, fontsize=16)

# 調整子圖之間的間距和底部空間
plt.subplots_adjust(bottom=0.1)  # 為legend留出空間

# 顯示圖表
plt.show()

In [ ]:
roe_collection_df = roe_collection.selected_stock_count_analysis()
roe_collection_df.reset_index(inplace=True)

# 建立顏色映射
colors = {
    '10': 'tab:blue',  # 藍色
    '15': 'orange',  # 橙色
    '20': 'tab:green',  # 綠色
    '25': 'tab:red'   # 紅色
}

# 提取年份和ROE閾值
roe_collection_df['Year'] = roe_collection_df['Strategy'].str.extract(r'roe_(\d)y')
roe_collection_df['ROE'] = roe_collection_df['Strategy'].str.extract(r'_(\d+)_')
roe_collection_df['PE'] = roe_collection_df['Strategy'].str.contains('有本益比')

# 創建圖表
fig, ax1 = plt.subplots(figsize=(10, 6))

# 獲取唯一的年份值
years = sorted(roe_collection_df['Year'].unique())

# 設定長條的寬度
bar_width = 0.15

# 計算每個年份組的位置
positions = np.arange(len(years))

# 儲存所有CAGR值用於設定y軸範圍
all_cagr_values = []

# 繪製有本益比的長條圖
bars = []  # 儲存長條物件用於之後設定legend
for i, roe_value in enumerate(['10', '15', '20', '25']):
    mask_with_pe = (roe_collection_df['ROE'] == roe_value) & (roe_collection_df['PE'])
    data_with_pe = [roe_collection_df[mask_with_pe & (roe_collection_df['Year'] == year)]['CAGR (%)'].values[0] 
                    if len(roe_collection_df[mask_with_pe & (roe_collection_df['Year'] == year)]) > 0 else 0 
                    for year in years]
    all_cagr_values.extend(data_with_pe)
    
    # 計算長條的位置
    x = positions + (i - 1.5) * bar_width

    # 繪製長條
    bar = ax1.bar(x, data_with_pe, bar_width, label=f'ROEN年平均 > {roe_value}%', color=colors[roe_value])
    bars.append(bar)

# 計算y軸範圍
y_min = min(all_cagr_values)
y_max = max(all_cagr_values)
y_margin = (y_max - y_min) * 0.1

# 設定y軸範圍
ax1.set_ylim(y_min - y_margin, y_max + y_margin)

# 設定x軸標籤
ax1.set_xticks(positions)
ax1.set_xticklabels([f'ROE{year}年平均' for year in years], fontsize=12)
ax1.set_title('台股_ROE變化_有本益比進出場策略的CAGR比較', fontsize=16)
ax1.set_xlabel('平均年份')
ax1.set_ylabel('CAGR (%)', fontsize=12)

# 添加網格線
ax1.grid(axis='y', linestyle='--', alpha=0.7)

# 添加legend到圖中
ax1.legend(ncol=4, fontsize=12, loc='upper center', bbox_to_anchor=(0.5, -0.1))

# 調整版面配置
plt.tight_layout()

# 顯示圖表
plt.show()

# fig.savefig('./img/圖 45台股有進出場條件策略ROE變化CAGR比較圖.svg', format='svg', bbox_inches='tight')

In [ ]:
def plot_strategy_comparison(df, compare='CAGR (%)', title=None, order_list=None, pattern=False, chart_type='bar'):
    """
    繪製策略比較的圖表
    
    Parameters:
    -----------
    df : pandas DataFrame
        包含策略資訊的數據框
    compare : str, default='CAGR (%)'
        要比較的指標欄位名稱
    title : str, default=None
        自訂圖表標題，若為None則使用預設標題
    order_list : list, default=None
        指定策略前綴的顯示順序，若為None則使用原始順序
    pattern : bool, default=False
        是否顯示子標籤模式
    chart_type : str, default='bar'
        圖表類型，可選 'bar' (柱狀圖) 或 'line' (折線圖)
    """
    import matplotlib.pyplot as plt
    import re
    
    # 取得所有策略名稱並分組
    strategies = df['Strategy'].tolist()
    strategy_pairs = {}
    
    for strategy in strategies:
        if '有本益比進出場' in strategy:
            prefix = strategy.replace('有本益比進出場', '')
            strategy_type = '有本益比'
        else:
            prefix = strategy.replace('無本益比進出場', '')
            strategy_type = '無本益比'
            
        if prefix not in strategy_pairs:
            strategy_pairs[prefix] = {}
        strategy_pairs[prefix][strategy_type] = df[df['Strategy'] == strategy][compare].values[0]

    # 如果有指定順序，重新排序strategy_pairs
    if order_list is not None:
        ordered_pairs = {k: strategy_pairs[k] for k in order_list if k in strategy_pairs}
        strategy_pairs = ordered_pairs

    # 設定圖表
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # 設定圖表位置
    x = range(len(strategy_pairs))
    width = 0.35
    
    # 根據圖表類型繪製
    if chart_type.lower() == 'bar':
        # 繪製柱狀圖
        rects1 = ax.bar([i - width/2 for i in x], 
                        [pair['有本益比'] for pair in strategy_pairs.values()],
                        width, label='有本益比')
        
        rects2 = ax.bar([i + width/2 for i in x], 
                        [pair['無本益比'] for pair in strategy_pairs.values()],
                        width, label='無本益比')
    elif chart_type.lower() == 'line':
        # 繪製折線圖
        line1 = ax.plot(x, 
                       [pair['有本益比'] for pair in strategy_pairs.values()],
                       'o-', linewidth=2, markersize=8, label='有本益比')
        
        line2 = ax.plot(x, 
                       [pair['無本益比'] for pair in strategy_pairs.values()],
                       's-', linewidth=2, markersize=8, label='無本益比')
        
        # ylim 使用整數，且從0開始
        ax.set_ylim(0, int(max([pair['有本益比'] for pair in strategy_pairs.values()] + 
                              [pair['無本益比'] for pair in strategy_pairs.values()]) * 1.1))
    else:
        raise ValueError("chart_type 必須是 'bar' 或 'line'")
    
    # 設定主要x軸標籤
    ax.set_ylabel(compare)
    ax.set_title(title if title else f'策略比較 - {compare}')
    ax.set_xticks(x)
    ax.set_xticklabels(strategy_pairs.keys(), rotation=45, ha='right', fontsize=14)

    # 處理pattern模式的子標籤
    if pattern:
        # 創建第二個x軸
        ax2 = ax.twiny()
        ax2.spines['top'].set_position(('axes', 1.0))
        
        # 找出所有前綴模式
        prefixes = list(strategy_pairs.keys())
        patterns = set()
        for prefix in prefixes:
            match = re.match(r'([^_]+_[^_]+)_.*', prefix)
            if match:
                patterns.add(match.group(1))
        
        patterns = sorted(list(patterns))
        pattern_positions = {}
        
        # 計算每個模式的平均位置
        for pattern in patterns:
            pattern_indices = [i for i, prefix in enumerate(prefixes) if prefix.startswith(pattern)]
            if pattern_indices:
                pattern_positions[pattern] = sum(pattern_indices) / len(pattern_indices)
        
        # 設定子標籤
        ax2.set_xlim(ax.get_xlim())
        ax2.set_xticks([pattern_positions[pattern] for pattern in patterns])
        ax2.set_xticklabels(patterns, rotation=0, ha='center', fontsize=14)
        
        # 添加垂直分隔線
        ax.grid(False)  # 關閉默認格線
        
        # 獲取y軸的範圍
        ymin, ymax = ax.get_ylim()
        
        # 在每個pattern的起始和結束位置添加垂直線
        for pattern in patterns:
            pattern_start = min([i for i, prefix in enumerate(prefixes) if prefix.startswith(pattern)]) - 0.5
            pattern_end = max([i for i, prefix in enumerate(prefixes) if prefix.startswith(pattern)]) + 0.5
            
            # 添加垂直分隔線
            ax.axvline(x=pattern_start, color='gray', 
                      linestyle='--', alpha=0.7, linewidth=0.5)
            ax.axvline(x=pattern_end, color='gray', 
                      linestyle='--', alpha=0.7, linewidth=0.5)
        
    ax.legend(loc='upper right')

    plt.grid(linestyle='--', alpha=0.7)
    plt.tight_layout()
    
    plt.show()

    fig.savefig(f'./img/{title}.svg', format='svg', bbox_inches='tight')

In [ ]:
plot_strategy_comparison(roe_collection_df, 
                         title='台股_ROE變化_有無本益比進出場策略比較_CAGR(%)',
                         order_list=['roe_3y_10_', 'roe_3y_15_', 'roe_3y_20_', 'roe_3y_25_', 'roe_3y_30_',
                                     'roe_4y_10_', 'roe_4y_15_', 'roe_4y_20_', 'roe_4y_25_', 'roe_4y_30_',
                                     'roe_5y_10_', 'roe_5y_15_', 'roe_5y_20_', 'roe_5y_25_', 'roe_5y_30_'],
                         pattern=True)

In [ ]:
plot_strategy_comparison(roe_collection_df,
                        compare='MDD (%)',
                        title='台股_ROE變化_有無本益比進出場策略比較_MDD(%)',
                        order_list=['roe_3y_10_', 'roe_3y_15_', 'roe_3y_20_', 'roe_3y_25_', 'roe_3y_30_',
                                     'roe_4y_10_', 'roe_4y_15_', 'roe_4y_20_', 'roe_4y_25_', 'roe_4y_30_',
                                     'roe_5y_10_', 'roe_5y_15_', 'roe_5y_20_', 'roe_5y_25_', 'roe_5y_30_'],
                        pattern=True
                        )

### 盈再率

In [ ]:
rr_value_cond = {}
no_rr_conds = (roe_15 & profit_cond & payout_ratio_cond & listed & hold_cond)[START_DATE:END_DATE]

for r in range(0, 81, 10): # 小於 0~40% (80%)

    rr_opt_df = rr.copy()
    rr_cond_opt = (rr_opt_df < (r/100))[START_DATE:END_DATE]

    rr_value_cond[f'盈再率小於_{r}%_無本益比進出場'] = (rr_cond_opt & no_rr_conds)[START_DATE:END_DATE]
    rr_value_cond[f'盈再率小於_{r}%_有本益比進出場'] = ((rr_cond_opt & no_rr_conds & pe_cond_entry_daily[START_DATE:END_DATE]).hold_until((~(rr_cond_opt & no_rr_conds)) | pe_cond_exit_daily[START_DATE:END_DATE]))

rr_collection = sim_conditions(rr_value_cond, resample='M', data=data)
rr_collection.selected_stock_count_analysis()

In [ ]:
# rr_collection.reports['盈再率小於_0%_有本益比進出場'].calc_returns_contrib()

In [ ]:
# rr_collection.reports['盈再率小於_40%_有本益比進出場'].calc_returns_contrib()

In [ ]:
rr_collection_df = rr_collection.selected_stock_count_analysis()
rr_collection_df.reset_index(inplace=True)

In [ ]:
plot_strategy_comparison(rr_collection_df,
                        compare='CAGR (%)',
                        title='台股_盈再率變化_有無本益比進出場策略比較_CAGR(%)',
                        order_list=['盈再率小於_0%_', '盈再率小於_10%_', '盈再率小於_20%_', '盈再率小於_30%_', '盈再率小於_40%_', '盈再率小於_50%_', '盈再率小於_60%_', '盈再率小於_70%_', '盈再率小於_80%_'])

In [ ]:
plot_strategy_comparison(rr_collection_df,
                        compare='MDD (%)',
                        title='台股_盈再率變化_有無本益比進出場策略比較_MDD(%)',
                        order_list=['盈再率小於_0%_', '盈再率小於_10%_', '盈再率小於_20%_', '盈再率小於_30%_', '盈再率小於_40%_', '盈再率小於_50%_', '盈再率小於_60%_', '盈再率小於_70%_', '盈再率小於_80%_']
                        )

### 配息率

In [ ]:
payout_strat_opt={}

payout_base_opt_conds = (roe_15 & rr_cond & profit_cond & hold_cond & listed)[START_DATE:END_DATE]

for j in range(0, 86, 5):

    dpr_cond_opt = (payout_ratio.rolling(3).min() >= j)[START_DATE:END_DATE]

    payout_all_opt_conds = payout_base_opt_conds & dpr_cond_opt

    payout_strat_opt[f'配息_三年至少{j}_無本益比進出場'] = (payout_all_opt_conds)[START_DATE:END_DATE]
    payout_strat_opt[f'配息_三年至少{j}_有本益比進出場'] = (payout_all_opt_conds & pe_cond_entry_daily[START_DATE:END_DATE]).hold_until(~payout_all_opt_conds | pe_cond_exit_daily[START_DATE:END_DATE])

payout_strat_opt_comb = sim_conditions(payout_strat_opt, resample='M', data=data)
payout_strat_opt_comb.selected_stock_count_analysis()

In [ ]:
# payout_strat_opt_comb.reports['配息_三年至少85_無本益比進出場'].display()

In [ ]:
payout_strat_opt_comb.reports['配息_三年至少85_無本益比進出場'].create_stacked_returns_plot()

In [ ]:
pr_collection_df = payout_strat_opt_comb.selected_stock_count_analysis()
pr_collection_df.reset_index(inplace=True)

In [ ]:
plot_strategy_comparison(pr_collection_df,
                        title='台股_配息_有無本益比進出場策略比較_CAGR(%)',
                        order_list=['配息_三年至少0_', '配息_三年至少5_', '配息_三年至少10_', '配息_三年至少15_', '配息_三年至少20_', '配息_三年至少25_', '配息_三年至少30_', '配息_三年至少35_', '配息_三年至少40_', '配息_三年至少45_', '配息_三年至少50_', '配息_三年至少55_'])

In [ ]:
plot_strategy_comparison(pr_collection_df,
                         compare='MDD (%)',
                         title='台股_配息_有無本益比進出場策略比較_MDD(%)',
                         order_list=['配息_三年至少0_', '配息_三年至少5_', '配息_三年至少10_', '配息_三年至少15_', '配息_三年至少20_', '配息_三年至少25_', '配息_三年至少30_', '配息_三年至少35_', '配息_三年至少40_', '配息_三年至少45_', '配息_三年至少50_', '配息_三年至少55_'],
                        )

In [ ]:
payout_strat_df = payout_strat_opt_comb.selected_stock_count_analysis()

In [ ]:
# reset index
payout_strat_df.reset_index(inplace=True)

In [ ]:
# # 從Strategy欄位提取n值
# payout_strat_df['n_value'] = payout_strat_df['Strategy'].str.extract(r'配息_(\d+)_').astype(int)

# # 按n_value排序
# payout_strat_df.sort_values(by='n_value', inplace=True)

# # 建立折線圖
# plt.figure(figsize=(10, 6))
# plt.plot(payout_strat_df['n_value'], payout_strat_df['CAGR (%)'], marker='o')

# plt.ylim(0, 16)

# # 設定圖表樣式
# plt.title('台股_2003~2024_配息率參數_其餘條件固定')
# plt.xlabel('年度股利支付率 (%)')
# plt.ylabel('CAGR (%)')
# plt.grid(True)

# # 顯示圖表
# plt.tight_layout()
# plt.show()

---

## 兩兩一組

In [ ]:
def plot_strategy_heatmap(df, compare='CAGR (%)', x_first=True, figsize=(14, 6), title=None, 
                         benchmark_param1=None, benchmark_param2=None, rep=None):
    """
    繪製策略熱力圖，當df['Min']==0時顯示灰色，並用白色框線標記Benchmark位置
    如果策略的日期不是從2003年開始，也會顯示灰色

    Parameters:
    -----------
    df : pandas DataFrame
        包含 'Strategy' 和比較欄位的數據框
    compare : str, default='CAGR (%)'
        要比較的欄位名稱，例如 'CAGR (%)' 或 'MDD (%)'
    x_first : bool, default=True
        True: 條件1為X軸，條件2為Y軸
        False: 條件1為Y軸，條件2為X軸
    figsize : tuple, default=(14, 6)
        圖形尺寸
    title : str, optional
        圖表標題，如果不指定則自動生成
    benchmark_param1 : float, optional
        指標1的基準值
    benchmark_param2 : float, optional
        指標2的基準值
    """
    
    if compare not in df.columns:
        raise ValueError(f"Column '{compare}' not found in DataFrame")
    
    param1_values = []
    param2_values = []
    not_start_2003 = []  # 儲存不是從2003年開始的策略
    
    # 從策略名稱中提取參數
    for strategy in df['Strategy']:
        date_index = rep.reports[strategy].position.index
        # print(date_index)
        
        # 檢查是否從2003年開始
        starts_from_2003 = False
        if len(date_index) > 0:
            first_date_str = str(date_index[0])
            if first_date_str.startswith('2003'):
                starts_from_2003 = True
                
        parts = strategy.split('_')
        # 處理可能帶有%的數值
        param1 = float(parts[1].replace('%', ''))
        param2 = float(parts[3].replace('%', ''))
        param1_values.append(param1)
        param2_values.append(param2)
        not_start_2003.append(not starts_from_2003)  # 記錄非2003開始的策略
    
    df['Param1'] = param1_values
    df['Param2'] = param2_values
    df['Not2003Start'] = not_start_2003  # 將結果添加到DataFrame
    
    condition1_name = df['Strategy'].iloc[0].split('_')[0]
    condition2_name = df['Strategy'].iloc[0].split('_')[2]

    if x_first:
        pivot_table = df.pivot(
            index='Param2',
            columns='Param1',
            values=compare
        )
        min_pivot = df.pivot(
            index='Param2',
            columns='Param1',
            values='Min'
        )
        not_2003_pivot = df.pivot(
            index='Param2',
            columns='Param1',
            values='Not2003Start'
        )
        pivot_table = pivot_table.reindex(index=sorted(pivot_table.index, reverse=True))
        min_pivot = min_pivot.reindex(index=sorted(min_pivot.index, reverse=True))
        not_2003_pivot = not_2003_pivot.reindex(index=sorted(not_2003_pivot.index, reverse=True))
        
        xlabel = f"{condition1_name} (%)"
        ylabel = f"{condition2_name} (%)"
        
        # 找出Benchmark在熱力圖中的位置
        if benchmark_param1 is not None and benchmark_param2 is not None:
            benchmark_x = pivot_table.columns.get_loc(benchmark_param1)
            benchmark_y = pivot_table.index.get_loc(benchmark_param2)
    else:
        pivot_table = df.pivot(
            index='Param1',
            columns='Param2',
            values=compare
        )
        min_pivot = df.pivot(
            index='Param1',
            columns='Param2',
            values='Min'
        )
        not_2003_pivot = df.pivot(
            index='Param1',
            columns='Param2',
            values='Not2003Start'
        )
        pivot_table = pivot_table.reindex(index=sorted(pivot_table.index, reverse=True))
        min_pivot = min_pivot.reindex(index=sorted(min_pivot.index, reverse=True))
        not_2003_pivot = not_2003_pivot.reindex(index=sorted(not_2003_pivot.index, reverse=True))
        
        xlabel = f"{condition2_name} (%)"
        ylabel = f"{condition1_name} (%)"
        
        # 找出Benchmark在熱力圖中的位置
        if benchmark_param1 is not None and benchmark_param2 is not None:
            benchmark_x = pivot_table.columns.get_loc(benchmark_param2)
            benchmark_y = pivot_table.index.get_loc(benchmark_param1)
    
    fig = plt.figure(figsize=figsize)
    
    mask = (min_pivot == 0)
    
    # 繪製熱力圖
    sns.heatmap(pivot_table,
                annot=True,
                annot_kws={'size': 14},
                fmt='.2f',
                cmap='coolwarm',
                cbar_kws={'label': compare},
                square=True,
                mask=None)
    
    # 在Min==0或不是從2003年開始的位置上覆蓋統一的灰色方塊
    for i in range(len(pivot_table.index)):
        for j in range(len(pivot_table.columns)):
            # 如果Min==0或不是從2003年開始，則繪製灰色方塊
            if mask.iloc[i, j] or (not_2003_pivot.iloc[i, j] == True):
                plt.gca().add_patch(plt.Rectangle((j, i), 1, 1, fill=True, color='#808080'))
    
    # 如果有指定Benchmark參數，繪製白色框線
    if benchmark_param1 is not None and benchmark_param2 is not None:
        plt.gca().add_patch(plt.Rectangle((benchmark_x, benchmark_y), 1, 1, 
                                        fill=False, edgecolor='white', linewidth=2))
    
    if title is None:
        title = f'{condition1_name} vs {condition2_name} 策略{compare}比較'
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    
    plt.tight_layout()
    plt.show()

    fig.savefig(f'./img/{title}.svg', format='svg', dpi=300)

### ROE & 盈再率

In [ ]:
roe_rr_opt_base_cond = payout_ratio_cond & profit_cond & hold_cond & listed

roe_rr_pe_opt_conds = {}

for r in range(0, 81, 10): # 小於 0~40% (80%)
    for p in range(10, 26, 5): # ROE 10~25%
        roe_5y_opt = roe.rolling(5).mean() > p
        rr_opt = rr < (r/100)

        roe_rr_opt_all_conds = (roe_5y_opt & rr_opt & roe_rr_opt_base_cond)[START_DATE:END_DATE]
        roe_rr_pe_opt_conds[f'ROE5年平均_{p}_盈再率_{r}__本益比進出場'] = (roe_rr_opt_all_conds & pe_cond_entry_daily[START_DATE:END_DATE]).hold_until(~roe_rr_opt_all_conds | pe_cond_exit_daily[START_DATE:END_DATE])

roe_rr_pe_opt_comb = sim_conditions(roe_rr_pe_opt_conds, resample='M', data=data)
# roe_rr_pe_opt_comb.selected_stock_count_analysis()

In [ ]:
fig = roe_rr_pe_opt_comb.plot_reps_stock_counts(['ROE5年平均_10_盈再率_0__本益比進出場', 'ROE5年平均_15_盈再率_0__本益比進出場', 'ROE5年平均_15_盈再率_40__本益比進出場'])
fig.savefig('./img/圖 61台股策略ROE與盈餘再投資率條件變化入選股數趨勢比較圖.svg', format='svg', dpi=300)

In [ ]:
# roe_rr_pe_opt_comb.reports['ROE5年平均_10_盈再率_80__本益比進出場'].plot_company_counts()

In [ ]:
# _,_=roe_rr_pe_opt_comb.reports['ROE5年平均_10_盈再率_0__本益比進出場'].calc_returns_contrib(top_n=5)

In [ ]:
roe_rr_opt_df = roe_rr_pe_opt_comb.selected_stock_count_analysis()
roe_rr_opt_df.reset_index(inplace=True)

In [ ]:
plot_strategy_heatmap(roe_rr_opt_df,
                      x_first=False,
                         title='2003~2024 台股_ROE5年平均_盈再率_本益比進出場_CAGR(%)',
                        figsize=(16, 6),
                        benchmark_param1=15,
                        benchmark_param2=40, rep = roe_rr_pe_opt_comb)

In [ ]:
plot_strategy_heatmap(roe_rr_opt_df,
                    compare='MDD (%)',
                    x_first=False,
                    title='2003~2024 台股_ROE5年平均_盈再率_本益比進出場_MDD(%)',
                    figsize=(16, 6),
                    benchmark_param1=15,
                    benchmark_param2=40, rep = roe_rr_pe_opt_comb)

### ROE & 本益比

In [ ]:
roe_pee_opt_base_conds = rr_cond & payout_ratio_cond & profit_cond & hold_cond & listed

roe_pee_opt_conds = {}

for roevalue in range(10, 26, 5): # ROE 10~25%
    for peevalue in range(8, 13, 2):  # 本益比

        roe_5y_opt = roe.rolling(5).mean() > roevalue
        pee_opt = (daily_pe < peevalue).resample('M').last()
        
        roe_pee_opt_all_conds = (roe_pee_opt_base_conds & roe_5y_opt)[START_DATE:END_DATE]

        roe_pee_opt_conds[f'ROE5年平均_{roevalue}%_本益比_{peevalue}__本益比進出場'] = (roe_pee_opt_all_conds & pee_opt[START_DATE:END_DATE]).hold_until((~roe_pee_opt_all_conds) | pe_cond_exit_daily[START_DATE:END_DATE])


roe_pee_opt_collecs = sim_conditions(roe_pee_opt_conds, resample='M', data=data)
roe_pee_opt_collecs.selected_stock_count_analysis()

In [ ]:
roe_pe_opt_df = roe_pee_opt_collecs.selected_stock_count_analysis()
roe_pe_opt_df.reset_index(inplace=True)

In [ ]:
plot_strategy_heatmap(roe_pe_opt_df,
                    title='2003~2024 台股_ROE5年平均_本益比進出場_CAGR(%)',
                    figsize=(18, 6),
                    benchmark_param1=15,
                    benchmark_param2=12,
                    rep = roe_pee_opt_collecs)

In [ ]:
plot_strategy_heatmap(roe_pe_opt_df,
                    compare='MDD (%)',
                    title='2003~2024 台股_ROE5年平均_本益比進出場_MDD(%)',
                    figsize=(18, 6),
                    benchmark_param1=15,
                    benchmark_param2=12,
                    rep = roe_pee_opt_collecs)

### ROE & 股利支付率

In [ ]:
roe_dpr_opt_base_conds = rr_cond & profit_cond & hold_cond & listed

roe_dpr_pe_opt_conds = {}

for i in range(10, 26, 5): # ROE 10~25%
    for n in range(0, 56, 5):  # 配息率 0~55%
        dpr_cond_rr_opt = payout_ratio.rolling(3).min() >= n
        roe_5y_opt = roe.rolling(5).mean() > i
        
        roe_dpr_opt_all_conds = (roe_dpr_opt_base_conds & roe_5y_opt & dpr_cond_rr_opt)[START_DATE:END_DATE]

        roe_dpr_pe_opt_conds[f'ROE5年平均_{i}_配息率_{n}__本益比進出場'] = (roe_dpr_opt_all_conds & pe_cond_entry_daily[START_DATE:END_DATE]).hold_until((~roe_dpr_opt_all_conds) | pe_cond_exit_daily[START_DATE:END_DATE])



roe_dpr_pe_opt_collecs = sim_conditions(roe_dpr_pe_opt_conds, resample='M', data=data)
roe_dpr_pe_opt_collecs.selected_stock_count_analysis()

In [ ]:
roe_dpr_pe_opt_df = roe_dpr_pe_opt_collecs.selected_stock_count_analysis()
roe_dpr_pe_opt_df.reset_index(inplace=True)

In [ ]:
plot_strategy_heatmap(roe_dpr_pe_opt_df,
                        x_first=False, 
                        title='2003~2024 台股_ROE5年平均_配息率_本益比進出場_CAGR(%)',
                        figsize=(16, 6),
                        benchmark_param1=15,
                        benchmark_param2=40,
                        rep = roe_dpr_pe_opt_collecs)

In [ ]:
plot_strategy_heatmap(roe_dpr_pe_opt_df,
                        x_first=False,
                        compare='MDD (%)',
                        title='2003~2024 台股_ROE5年平均_配息率_本益比進出場_MDD(%)',
                        figsize=(16, 6),
                        benchmark_param1=15,
                        benchmark_param2=40,
                        rep = roe_dpr_pe_opt_collecs)

### 盈再率 & 股利支付率

In [ ]:
rr_dpr_opt_base_conds = roe_15 & profit_cond & hold_cond & listed

rr_dpr_pe_opt_conds = {}

for i in range(0, 56, 5): 
    for n in range(0, 81, 10):
        dpr_cond_rr_opt = payout_ratio.rolling(3).min() >= i
        rr_dpr_opt_all_conds = (rr_dpr_opt_base_conds & (rr < (n/100)) & dpr_cond_rr_opt)[START_DATE:END_DATE]

        rr_dpr_pe_opt_conds[f'盈再率_{n}_配息率_{i}_本益比進出場'] = (rr_dpr_opt_all_conds & pe_cond_entry_daily[START_DATE:END_DATE]).hold_until((~rr_dpr_opt_all_conds) | pe_cond_exit_daily[START_DATE:END_DATE])



rr_dpr_pe_opt_collecs = sim_conditions(rr_dpr_pe_opt_conds, resample='M', data=data)
rr_dpr_pe_opt_collecs.selected_stock_count_analysis()

In [ ]:
fig = rr_dpr_pe_opt_collecs.plot_reps_stock_counts(['盈再率_0_配息率_30_本益比進出場', '盈再率_0_配息率_50_本益比進出場', '盈再率_40_配息率_40_本益比進出場'])
fig.savefig('./img/圖 68台股策略盈餘再投資率與股利支付率變化入選股數趨勢比較圖.svg', format='svg', dpi=300)

In [ ]:
rr_dpr_pe_opt_df = rr_dpr_pe_opt_collecs.selected_stock_count_analysis()
rr_dpr_pe_opt_df.reset_index(inplace=True)

In [ ]:
rr_dpr_pe_opt_df.loc[7]

In [ ]:
plot_strategy_heatmap(rr_dpr_pe_opt_df,
                        x_first=False, 
                        title='2003~2024 台股_盈再率_配息率_本益比進出場_CAGR(%)',
                        figsize=(16, 10),
                        benchmark_param1=40,
                        benchmark_param2=40,
                        rep = rr_dpr_pe_opt_collecs)

In [ ]:
plot_strategy_heatmap(rr_dpr_pe_opt_df,
                        x_first=False,
                        compare='MDD (%)',
                        title='2003~2024 台股_盈再率_配息率_本益比進出場_MDD(%)',
                        figsize=(16, 10),
                        benchmark_param1=40,
                        benchmark_param2=40,
                        rep = rr_dpr_pe_opt_collecs)

### 盈再率 & 本益比

In [ ]:
rr_pee_opt_base_conds = roe_15 & payout_ratio_cond & profit_cond & hold_cond & listed

rr_pee_opt_conds = {}

for rrvalue in range(0, 81, 10): # 盈再率
    for peevalue in range(8, 13, 2):  # 本益比

        pee_opt = (daily_pe < peevalue).resample('M').last()
        rrvalue_opt = rr.copy() < (rrvalue/100)
        
        rr_pee_opt_all_conds = (rr_pee_opt_base_conds & rrvalue_opt)[START_DATE:END_DATE]

        rr_pee_opt_conds[f'盈再率小於_{rrvalue}%_本益比_{peevalue}__本益比進出場'] = (rr_pee_opt_all_conds & pee_opt[START_DATE:END_DATE]).hold_until((~rr_pee_opt_all_conds) | pe_cond_exit_daily[START_DATE:END_DATE])

rr_pee_collecs = sim_conditions(rr_pee_opt_conds, resample='M', data=data)
rr_pee_collecs.selected_stock_count_analysis()

In [ ]:
fig = rr_pee_collecs.plot_reps_stock_counts(["盈再率小於_0%_本益比_12__本益比進出場", "盈再率小於_40%_本益比_12__本益比進出場"])
fig.savefig('./img/圖 71台股策略盈餘再投資率與本益比進場條件變化入選股數比較圖.svg', format='svg', dpi=300)

In [ ]:
rr_pee_opt_df = rr_pee_collecs.selected_stock_count_analysis()
rr_pee_opt_df.reset_index(inplace=True)

plot_strategy_heatmap(rr_pee_opt_df,
                        x_first=True, 
                        title='2003~2024 台股_盈再率_本益比進出場_CAGR(%)',
                        figsize=(16, 6),
                        benchmark_param1=40,
                        benchmark_param2=12,
                        rep = rr_pee_collecs)

In [ ]:
plot_strategy_heatmap(rr_pee_opt_df,
                        x_first=True, 
                        compare='MDD (%)',
                        title='2003~2024 台股_盈再率_本益比進出場_MDD(%)',
                        figsize=(16, 6),
                        benchmark_param1=40,
                        benchmark_param2=12,
                        rep = rr_pee_collecs)

### 股利支付率 & 本益比

In [ ]:
dpr_pee_opt_base_conds = roe_15 & rr_cond & profit_cond & hold_cond & listed

dpr_pee_opt_conds = {}

for povalue in range(0, 56, 5): # 配息率
    for peevalue in range(8, 13, 2):  # 本益比

        pee_opt = (daily_pe < peevalue).resample('M').last()
        dpr_cond_rr_opt = payout_ratio.rolling(3).min() >= povalue
        
        dpr_pee_opt_all_conds = (dpr_pee_opt_base_conds & dpr_cond_rr_opt)[START_DATE:END_DATE]

        dpr_pee_opt_conds[f'配息率_{povalue}%_本益比_{peevalue}__本益比進出場'] = (dpr_pee_opt_all_conds & pee_opt[START_DATE:END_DATE]).hold_until((~dpr_pee_opt_all_conds) | pe_cond_exit_daily[START_DATE:END_DATE])

dpr_pee_opt_collecs = sim_conditions(dpr_pee_opt_conds, resample='M', data=data)
dpr_pee_opt_collecs.selected_stock_count_analysis()

In [ ]:
dpr_pee_opt_df = dpr_pee_opt_collecs.selected_stock_count_analysis()
dpr_pee_opt_df.reset_index(inplace=True)

In [ ]:
plot_strategy_heatmap(dpr_pee_opt_df,
                        x_first=True, 
                        compare='CAGR (%)',
                        title='2003~2024 台股_配息率_本益比進出場_CAGR(%)',
                        figsize=(16, 6),
                        benchmark_param1=40,
                        benchmark_param2=12,
                        rep = dpr_pee_opt_collecs)

In [ ]:
plot_strategy_heatmap(dpr_pee_opt_df,
                        x_first=True, 
                        compare='MDD (%)',
                        title='2003~2024 台股_配息率_本益比進出場_MDD(%)',
                        figsize=(16, 6),
                        benchmark_param1=40,
                        benchmark_param2=12,
                        rep = dpr_pee_opt_collecs)

In [ ]:
# # 從Strategy欄位中提取盈再率和配息率的值
# reinvestment_rates = []
# dividend_rates = []

# for strategy in rr_dpr_pe_opt_df['Strategy']:
#     # 分割字串並提取數值
#     parts = strategy.split('_')
#     reinvestment_rate = float(parts[1])  # 盈再率的值
#     dividend_rate = float(parts[3])      # 配息率的值
    
#     reinvestment_rates.append(reinvestment_rate)
#     dividend_rates.append(dividend_rate)

# # 將提取的值添加到DataFrame中
# rr_dpr_pe_opt_df['Reinvestment_Rate'] = reinvestment_rates
# rr_dpr_pe_opt_df['Dividend_Rate'] = dividend_rates

# # 創建樞紐表
# pivot_table = rr_dpr_pe_opt_df.pivot(
#     index='Reinvestment_Rate',
#     columns='Dividend_Rate',
#     values='CAGR (%)'
# )

# # 設置圖形大小
# plt.figure(figsize=(12, 5))

# # 繪製熱力圖
# sns.heatmap(pivot_table, 
#             annot=True,           # 顯示數值
#             fmt='.2f',            # 數值格式化為兩位小數
#             cmap='coolwarm',      # 使用紅黃藍色階，_r表示顏色反轉
#             cbar_kws={'label': 'CAGR (%)'},  # 設置色階標籤
#             square=True
# )

# # 設置標題和軸標籤
# plt.title('CAGR (%) Heatmap by Reinvestment Rate and Dividend Rate')
# plt.xlabel('股利支付率 (%)')
# plt.ylabel('盈再率 (%)')

# # 調整布局
# plt.tight_layout()

# # 顯示圖形
# plt.show()